# 把价格和财报变成可重复的研究输入

这篇教程沿一次研究准备工作展开：取价格，理解缺失与价格尺度，再把当时可见的财报接进特征，保存后重复使用。表格与代码输出沿用已执行的 Tushare 小样本；本次调整教学顺序与展示，未重新采集。


## 1. 确定这次要取什么

先用 `000001.SZ`、`600000.SH`、`600519.SH` 最后六个交易日的行情检查读法。后面加入财报时，使用已有的 `000001.SZ`、`600000.SH` 两证券完整来源样本，最后把价格、限价和财报组成一张按证券与日期对齐的特征面板。

第一份行情覆盖 2025-09-01—2026-08-31 的 242 个交易日。初始化把它选定为 `DATA` 和 `SNAPSHOT`，将最后六个交易日记为 `DAYS`。这次实验一直沿用这个具体版本，后面新数据发布也不会改变已经选好的输入。

先取小表把列、单位和日历对上；需要更长窗口时，再从已存历史扩大读取范围。初始化路径放在下面，阅读结果时可以直接进入第 2 节。


In [1]:
from pathlib import Path
import sys
repo = Path.cwd() / 'axiom-docs'
if not repo.is_dir():
    repo = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'examples/tutorial_support.py').exists())
sys.path.insert(0, str(repo / 'examples'))
from tutorial_support import *  # pinned local data; writes use TEMP


固定版本与所选交易日可在这里核对。完整标识用来复现，后续调用沿用变量名。


In [2]:
print('Snapshot:', SNAPSHOT)
print('证券:', SYMBOL)
print('Snapshot已存开放日历:', len(CALENDAR), '日；范围:', CALENDAR[0], '至', CALENDAR[-1])
print('本查询仅选择末尾', len(DAYS), '日:', DAYS)
print('本例只读取本地数据，供应商请求为 0。')

Snapshot: s_c3faabc9ec124597e91701af33ebac563cd386eff776ac467f2a2462340bff2b
证券: cnstock.000001.SZ.19910403
Snapshot已存开放日历: 242 日；范围: 2025-09-01 至 2026-08-31
本查询仅选择末尾 6 日: ('2026-08-24', '2026-08-25', '2026-08-26', '2026-08-27', '2026-08-28', '2026-08-31')
本例只读取本地数据，供应商请求为 0。


## 2. 取出第一张可以计算的表

先从一张小表开始。我们从已经保存的一年行情中，取出 `000001.SZ` 最后六个交易日的开盘价、收盘价和成交量。先看清一只证券，再把同样的查询扩到三只。

本节使用初始化中既定的收盘后口径：每个交易日 20:30 的 cutoff，以及 `best_effort_vendor_v1`。加入财报时，再看这个时间口径怎样影响可见结果。


In [3]:
q = QuerySpec('market_daily', ('open','close','volume_shares'), (SYMBOL,),
    DAYS, 'best_effort_vendor_v1', {d:d+'T20:30:00+08:00' for d in DAYS})
batch = DATA.read(snapshot=SNAPSHOT, query=q)
display(batch.frame)
print('单位:', {k:v['unit'] for k,v in batch.field_meta.items()})

,security_id,session,open,close,volume_shares
0,cnstock.000001.SZ.19910403,2026-08-24,11.38,11.56,119902495
1,cnstock.000001.SZ.19910403,2026-08-25,11.57,11.59,99488115
2,cnstock.000001.SZ.19910403,2026-08-26,11.55,11.73,116728878
3,cnstock.000001.SZ.19910403,2026-08-27,11.7,11.59,97570170
4,cnstock.000001.SZ.19910403,2026-08-28,11.54,11.65,83812624
5,cnstock.000001.SZ.19910403,2026-08-31,11.64,11.72,90885655


单位: {'open': 'CNY/share', 'close': 'CNY/share', 'volume_shares': 'shares'}


一行对应一个交易日。open 和 close 的单位是元/股，volume_shares 已经是股数。想多读历史，扩大 `DAYS`；想多取列，修改 `fields`。

接着只取 close，把证券集合扩到 `000001.SZ`、`600000.SH`、`600519.SH`，再按证券和交易日展开成矩阵。`query()` 沿用刚才的交易日与收盘后口径。


In [5]:
panel = DATA.read(snapshot=SNAPSHOT, query=query(('close',),
    symbols=tuple(IDS[c] for c in ('000001.SZ','600000.SH','600519.SH'))))
display(panel.frame.pivot(index='session',columns='security_id',values='close'))

security_id,cnstock.000001.SZ.19910403,cnstock.600000.SH.19991110,cnstock.600519.SH.20010827
session,,,
2026-08-24,11.56,9.22,1304.66
2026-08-25,11.59,9.08,1304.0
2026-08-26,11.73,9.21,1302.8
2026-08-27,11.59,9.07,1292.3
2026-08-28,11.65,9.0,1297.4
2026-08-31,11.72,9.16,1299.52


现在有了六行日线和三证券收盘矩阵。矩阵只是将 `(security_id, session)` 展开，接下来还要决定用于计算的价格尺度。

需要加入新字段时，可以展开下面的字段字典，查看规范名和单位。这个行情版本已有价格、成交量和因子；收入字段显示“支持但该 Snapshot 未接入”。因此财报阶段会换到两证券完整来源样本。


In [4]:
fact_dictionary = DATA.dictionary(snapshot=SNAPSHOT,
    domains=('market_daily', 'adjustment_factors', 'financial_events'))
dictionary_frame = pd.DataFrame(fact_dictionary['fields'])
selected_dictionary = dictionary_frame[
    dictionary_frame.endpoint.isin(('daily', 'adj_factor', 'income')) &
    dictionary_frame.canonical_field.isin(('open', 'close', 'volume_shares', 'factor', 'total_revenue'))]
display(selected_dictionary[['domain','raw_field','canonical_field','meaning','raw_unit','unit','conversion','status']]
    .rename(columns={'domain':'域','raw_field':'Raw字段','canonical_field':'规范字段','meaning':'含义',
                     'raw_unit':'来源单位','unit':'规范单位','conversion':'转换','status':'当前状态'}))
assert selected_dictionary.loc[selected_dictionary.canonical_field.eq('total_revenue'), 'snapshot_available'].eq(False).all()
print('字典只读合同与来源映射；实际值、PIT 可见性和缺失原因仍看查询结果。')

,域,Raw字段,规范字段,含义,来源单位,规范单位,转换,当前状态
1,adjustment_factors,adj_factor,factor,供应商累计复权因子；调整价使用因子比,dimensionless,dimensionless,None,Snapshot已声明
32,financial_events,total_revenue,total_revenue,营业总收入,CNY,CNY,None,支持但该Snapshot未接入
34,market_daily,close,close,未复权收盘价（指数域为指数点位）,CNY/share,CNY/share,None,Snapshot已声明
39,market_daily,open,open,未复权开盘价,CNY/share,CNY/share,None,Snapshot已声明
47,market_daily,vol,volume_shares,成交股数,hundred-share lots,shares,"{'factor': '100', 'from': 'hundred-share lots'...",Snapshot已声明


字典只读合同与来源映射；实际值、PIT 可见性和缺失原因仍看查询结果。


## 3. 把价格变成可比较的输入

刚才的六天没有空值。把窗口拉长后，会遇到休市、停牌和上市前日期；先保留空值，再查看 `Data.states` 给出的原因。

下面的 `002185.SZ` 在 2025-09-25 停牌，09-27 是周末，两个日期都没有可用价格。`603418.SH` 的例子还区分历史当时未知的上市身份与今天回顾时已知的上市日期。


In [7]:
missing = sorted(AUDIT['missing_market_samples'], key=lambda r: (r['security_id'], r['session']))[0]
closed_day = '2025-09-27'
state_query = query(symbols=(missing['security_id'],), sessions=(missing['session'], closed_day))
state_batch = DATA.states(snapshot=SNAPSHOT, query=state_query)
display(state_batch.frame)
assert set(state_batch.frame.market_state) == {'suspended', 'calendar_closed'}
master_rows = [r for p in MANIFEST['domains']['security_master']['partitions']
               for r in DATA.store.read_partition(p).to_pylist()]
new_listing = next(r for r in master_rows if str(r['listing_date']) > '2025-09-01')
past_knowledge = DATA.states(snapshot=SNAPSHOT,
    query=query(symbols=(new_listing['security_id'],), sessions=('2025-09-01',)))
display(past_knowledge.frame)
print('原日期 cutoff 尚不能看到后来上市身份；再用实际当前知识回顾该历史日期。')
pre_listing = DATA.states(snapshot=SNAPSHOT,
    query=query(symbols=(new_listing['security_id'],), sessions=('2025-09-01',),
                policy='operational_pit_v1', common_cutoff=datetime.now(timezone.utc).isoformat()))
display(pre_listing.frame)
assert pre_listing.frame.market_state.iloc[0] == 'not_listed'
print('全范围缺价:', AUDIT['missing_market_cells'],
      '有停牌证据:', AUDIT['known_suspension_missing_cells'], '未解释:',
      AUDIT['missing_market_cells'] - AUDIT['known_suspension_missing_cells'])


,security_id,session,market_state,close
0,cnstock.002185.SZ.20071120,2025-09-25,suspended,<NA>
1,cnstock.002185.SZ.20071120,2025-09-27,calendar_closed,None


,security_id,session,market_state,close
0,cnstock.603418.SH.20250923,2025-09-01,unknown_status,<NA>


原日期 cutoff 尚不能看到后来上市身份；再用实际当前知识回顾该历史日期。


,security_id,session,market_state,close
0,cnstock.603418.SH.20250923,2025-09-01,not_listed,None


全范围缺价: 158 有停牌证据: 158 未解释: 0


有价格时也要留意状态。两证券完整来源样本在 2026-08-31 可以读到 close，但停复牌来源为空，状态仍是 unknown。研究可以按已声明口径使用价格，执行是否准入由 Engine 的 profile 另行决定。

下面展开上市/退市来源和状态核对；日常算特征先看逐键价格与缺失原因。


In [8]:
# Production listing and status facts come from the pinned Tushare Snapshot.
listing_manifest = UNIFIED.store.load_snapshot(UNIFIED_SNAPSHOT)
master_domain = listing_manifest['domains']['security_master']
assert master_domain['source_profile']['id'].startswith('tushare.')
master = pd.DataFrame([r for part in master_domain['partitions']
    for r in UNIFIED.store.read_partition(part).to_pylist()])
listing_columns = [c for c in ('security_id','listing_date','delisting_date',
    'vendor_delist_date','list_status','first_observed_at') if c in master.columns]
display(master[listing_columns].head(8))
print('生产上市与退市来源:', master_domain['source_profile']['id'])

,security_id,listing_date,delisting_date,vendor_delist_date,list_status,first_observed_at
0,cnstock.600000.SH.19991110,1999-11-10,None,None,L,2026-09-28 13:16:33.564276+00:00
1,cnstock.000001.SZ.19910403,1991-04-03,None,None,L,2026-09-28 13:16:37.320913+00:00


生产上市与退市来源: tushare.local.stock_basic.v1


In [9]:
if 'listing_events' in listing_manifest['domains']:
    event_domain = listing_manifest['domains']['listing_events']
    assert event_domain['source_profile']['id'].startswith('tushare.')
    events = pd.DataFrame([r for part in event_domain['partitions']
        for r in UNIFIED.store.read_partition(part).to_pylist()])
    if not events.empty:
        event_columns = [c for c in ('security_id','listing_date','delisting_date',
            'event_type','first_observed_at') if c in events.columns]
        delisted_events = (events.loc[events.delisting_date.notna()]
            if 'delisting_date' in events else events)
        display(delisted_events[event_columns].head(8))
        print('规范上市事件来源:', event_domain['source_profile']['id'])
state_day = '2026-08-31'
state_ids = (IDS['000001.SZ'], IDS['600000.SH'])

,security_id,listing_date,delisting_date,event_type,first_observed_at
0,cnstock.000508.SZ.19930430,1993-04-30,1999-07-12,delisting,2026-09-28 13:16:38.388413+00:00
1,cnstock.600625.SH.19930106,1993-01-06,2001-04-23,delisting,2026-09-28 13:16:34.649002+00:00
2,cnstock.000588.SZ.19960123,1996-01-23,2001-06-15,delisting,2026-09-28 13:16:38.388413+00:00
3,cnstock.000015.SZ.19920625,1992-06-25,2001-10-22,delisting,2026-09-28 13:16:38.388413+00:00
4,cnstock.000556.SZ.19940525,1994-05-25,2002-05-29,delisting,2026-09-28 13:16:38.388413+00:00
5,cnstock.000003.SZ.19910703,1991-07-03,2002-06-14,delisting,2026-09-28 13:16:38.388413+00:00
6,cnstock.000675.SZ.19961226,1996-12-26,2002-08-20,delisting,2026-09-28 13:16:38.388413+00:00
7,cnstock.000689.SZ.19961231,1996-12-31,2002-09-05,delisting,2026-09-28 13:16:38.388413+00:00


规范上市事件来源: tushare.stock_basic.listing_events.v1


In [10]:
state_q = QuerySpec('market_daily', ('close',), state_ids, (state_day,),
    'best_effort_vendor_v1', {state_day:state_day+'T20:30:00+08:00'})
production_states = UNIFIED.states(snapshot=UNIFIED_SNAPSHOT, query=state_q)
display(production_states.frame)
past_states = UNIFIED.states(snapshot=UNIFIED_SNAPSHOT,
    query=replace(state_q, pit_policy='operational_pit_v1'))
display(past_states.frame)
assert past_states.frame.market_state.eq('unknown_status').all()
print('交易状态依据留存供应商事实和日历推导；外部交叉校验只能提示冲突，不改写生产值。')
print('严格历史 cutoff 早于原 Raw 接收，状态仍未知。')

,security_id,session,market_state,close
0,cnstock.000001.SZ.19910403,2026-08-31,unknown_status,11.72
1,cnstock.600000.SH.19991110,2026-08-31,unknown_status,9.16


,security_id,session,market_state,close
0,cnstock.000001.SZ.19910403,2026-08-31,unknown_status,<NA>
1,cnstock.600000.SH.19991110,2026-08-31,unknown_status,<NA>


交易状态依据留存供应商事实和日历推导；外部交叉校验只能提示冲突，不改写生产值。
严格历史 cutoff 早于原 Raw 接收，状态仍未知。


### 让同一证券的历史价格处在同一尺度

要从 close 计算收益，先将窗口里的原价和复权因子放到共同锚点：

**调整价(t) = 原价(t) × factor(t) / factor(锚点)。**

本例将锚点设在 2026-08-31，价格和因子使用同一个 cutoff。六天的累计因子都为 139.008，所以因子比都为 1，调整后的 close 与原价相等。第一行没有前一观察，`observed_return_1d` 留空。

较长窗口里因子可能变化，比较的是因子比，而不是累计因子是否等于 1。模拟某个历史决策时，锚点与可见因子也要固定在那个时点。


In [11]:
cutoff = DAYS[-1] + 'T20:30:00+08:00'
prices = DATA.read(snapshot=SNAPSHOT, query=query(common_cutoff=cutoff))
factors = DATA.read(snapshot=SNAPSHOT, query=query(('factor',),
    domain='adjustment_factors', common_cutoff=cutoff))
adjusted = adjust_prices(prices, factors, fields=('close',),
    factor_field='factor', anchor_session=DAYS[-1])
research = adjusted.frame.copy()
research['observed_return_1d'] = research.groupby('security_id').close.pct_change(fill_method=None)
factor_table = factors.frame.rename(columns={'factor':'cumulative_factor'})
research = research.merge(prices.frame.rename(columns={'close':'original_close'}), on=['security_id','session'])
research = research.merge(factor_table, on=['security_id','session'])
anchor_factor = float(factors.frame.loc[factors.frame.session.eq(DAYS[-1]), 'factor'].iloc[0])
research['factor_ratio'] = research.cumulative_factor / anchor_factor
display(research)
print('派生合同:', adjusted.context['derivation'])
assert research.observed_return_1d.iloc[0] is pd.NA or pd.isna(research.observed_return_1d.iloc[0])


,security_id,session,close,observed_return_1d,original_close,cumulative_factor,factor_ratio
0,cnstock.000001.SZ.19910403,2026-08-24,11.56,NaN,11.56,139.008,1.0
1,cnstock.000001.SZ.19910403,2026-08-25,11.59,0.002595,11.59,139.008,1.0
2,cnstock.000001.SZ.19910403,2026-08-26,11.73,0.012079,11.73,139.008,1.0
3,cnstock.000001.SZ.19910403,2026-08-27,11.59,-0.011935,11.59,139.008,1.0
4,cnstock.000001.SZ.19910403,2026-08-28,11.65,0.005177,11.65,139.008,1.0
5,cnstock.000001.SZ.19910403,2026-08-31,11.72,0.006009,11.72,139.008,1.0


派生合同: {'recipe_version': 'common_anchor_price_v1', 'formula': 'price_t * factor_t / factor_anchor', 'factor_field': 'factor', 'anchor_session': '2026-08-31', 'decision_session': '2026-08-31', 'decision_cutoff': '2026-08-31T12:30:00+00:00', 'factor_domain': 'adjustment_factors', 'factor_contract_id': 'local.adjustment_factors.terminal_observation.v1', 'factor_source_profile_id': 'tushare.local.adj_factor.v1', 'factor_reader_version': 'local_reader_v5', 'price_query': {'fields': ['close'], 'symbols': ['cnstock.000001.SZ.19910403'], 'sessions': ['2026-08-24', '2026-08-25', '2026-08-26', '2026-08-27', '2026-08-28', '2026-08-31'], 'pit_policy': 'best_effort_vendor_v1', 'cutoff_by_session': {'2026-08-24': '2026-08-31T20:30:00+08:00', '2026-08-25': '2026-08-31T20:30:00+08:00', '2026-08-26': '2026-08-31T20:30:00+08:00', '2026-08-27': '2026-08-31T20:30:00+08:00', '2026-08-28': '2026-08-31T20:30:00+08:00', '2026-08-31': '2026-08-31T20:30:00+08:00'}, 'purpose': 'decision_facts', 'price_basis': 'u

## 4. 给价格加一列财报信息

接下来想把收入放到价格旁边。一年行情样本没有财务域，因此改用已有的两证券完整来源样本：行情窗口是 2026-06—08 月，财务另有前置报告期。

先读取 `600000.SH` 的合并收入表，字段为 `total_revenue` 和 `parent_net_income`，单位都是 CNY。`EventQuery` 按报告期返回记录；这一步用当前知识查看样本里的报告，下一节再回到历史 cutoff。域清单与证券范围可以展开核对。


In [12]:
# The unified Snapshot combines retained original Tushare Raw and dated vendor membership.
FULL_ROOT = UNIFIED_ROOT
full = UNIFIED
FULL_SNAPSHOT = UNIFIED_SNAPSHOT
full_manifest = full.store.load_snapshot(FULL_SNAPSHOT)
full_scope = json.loads((REPO / 'delivery/csi1800_tushare_only_v4_202606_202608.scope.json').read_text())
bank_id = full_scope['identity_map']['600000.SH']
display(pd.DataFrame([{'domain':name,
    'rows':sum(p['rows'] for p in segment['partitions']),
    'raw_receipts':len(segment['raw_batch_ids'])}
    for name, segment in full_manifest['domains'].items()]))
fin_domains = ('financial_events', 'balance_sheet_events', 'cash_flow_events',
               'financial_indicator_events', 'corporate_actions',
               'top_holders_reports', 'price_limits')
assert all(name in full_manifest['domains'] for name in fin_domains), 'complete-source publication is required'

,domain,rows,raw_receipts
0,adjustment_factors,130,65
1,balance_sheet_events,13,20
2,cash_flow_events,10,20
3,corporate_actions,20,277
4,financial_events,14,20
5,financial_indicator_events,12,6
6,listing_events,5888,6
7,market_daily,130,65
8,price_limits,130,65
9,security_master,2,6


In [13]:
source_domains = fin_domains + ('listing_events', 'universe_membership')
assert all(name in full_manifest['domains'] for name in source_domains)
assert all(full_manifest['domains'][name]['source_profile']['id'].startswith('tushare.')
    for name in source_domains), 'production domains must use retained Tushare Raw'
inventory = []
for name in source_domains:
    segment = full_manifest['domains'][name]
    inventory.append({'domain':name, 'rows':sum(p['rows'] for p in segment['partitions']),
        'partitions':len(segment['partitions']), 'raw_receipts':len(segment['raw_batch_ids']),
        'source_profile':segment['source_profile']['id'],
        'native_key':', '.join(segment['contract']['logical_key'])})
display(pd.DataFrame(inventory))
print('统一完整源根:', FULL_ROOT, '固定版本:', FULL_SNAPSHOT)
market_ids = sorted({r['security_id'] for p in full_manifest['domains']['market_daily']['partitions']
    for r in full.store.read_partition(p).to_pylist()})

,domain,rows,partitions,raw_receipts,source_profile,native_key
0,financial_events,14,3,20,tushare.local.income.v1,"security_id, endpoint, report_type, report_period"
1,balance_sheet_events,13,3,20,tushare.local.balancesheet.v1,"security_id, endpoint, report_type, report_period"
2,cash_flow_events,10,3,20,tushare.local.cashflow.v1,"security_id, endpoint, report_type, report_period"
3,financial_indicator_events,12,2,6,tushare.local.fina_indicator.v1,"security_id, endpoint, report_period"
4,corporate_actions,20,3,277,tushare.local.dividend.v1,"security_id, report_period, announcement_date,..."
5,top_holders_reports,12,2,2,tushare.local.top10_holders.v1,"security_id, report_period"
6,price_limits,130,3,65,tushare.local.stk_limit.v1,"security_id, session"
7,listing_events,5888,1,6,tushare.stock_basic.listing_events.v1,"security_id, event_type"
8,universe_membership,3874,1,12,tushare.index_weight.dated_membership.v1,membership_id


统一完整源根: /Users/liuming/Documents/axiom/data/csi1800_tushare_only_v4_202606_202608 固定版本: s_a0db6028118997d93b9f70dff852b701b030b152a30b2e8d325c08ad64931049


In [14]:
print('采集范围:', full_scope['start_session'], '至', full_scope['end_session'],
      '；市场证券:', ', '.join(market_ids), '；成员候选身份数:', len(full_scope['symbols']))
print('此 Snapshot 同时含 Tushare 成员观测；供应商源由留存原始 Raw 离线回放，无新在线请求。')

采集范围: 2026-06-01 至 2026-08-31 ；市场证券: cnstock.000001.SZ.19910403, cnstock.600000.SH.19991110 ；成员候选身份数: 1905
此 Snapshot 同时含 Tushare 成员观测；供应商源由留存原始 Raw 离线回放，无新在线请求。


In [15]:
finance_domain = full_manifest['domains']['financial_events']
finance_rows = [row for part in finance_domain['partitions']
                for row in full.store.read_partition(part, symbols=(bank_id,)).to_pylist()
                if row['security_id'] == bank_id and row['endpoint'] == 'income'
                and row['report_type'] == '1']
assert finance_rows, 'expected real Tushare consolidated income observations'
periods = sorted({str(row['report_period']) for row in finance_rows})
cutoff_now = datetime.now(timezone.utc).isoformat()
income_query = EventQuery('financial_events', ('total_revenue','parent_net_income'),
    (bank_id,), periods[0], periods[-1], cutoff_now, 'operational_pit_v1',
    'report_period', {'endpoint':'income','report_type':'1'})
income_events = full.events(snapshot=FULL_SNAPSHOT, query=income_query)
display(income_events.frame)
assert len(income_events.frame) > 0
assert income_events.field_meta['total_revenue']['unit'] == 'CNY'
display(pd.DataFrame(income_events.field_meta['total_revenue']['by_key']))
print('原生报告键:', finance_domain['contract']['logical_key'])
print('Reader 先按 cutoff 选可见 revision，再按报告期和 endpoint 投影。')


,security_id,endpoint,report_type,report_period,total_revenue,parent_net_income
0,cnstock.600000.SH.19991110,income,1,2024-12-31,170748000000.0,45257000000.0
1,cnstock.600000.SH.19991110,income,1,2025-03-31,45922000000.0,17598000000.0
2,cnstock.600000.SH.19991110,income,1,2025-06-30,90559000000.0,29737000000.0
3,cnstock.600000.SH.19991110,income,1,2025-09-30,132280000000.0,38819000000.0
4,cnstock.600000.SH.19991110,income,1,2025-12-31,173964000000.0,50017000000.0
5,cnstock.600000.SH.19991110,income,1,2026-03-31,46573000000.0,17861000000.0
6,cnstock.600000.SH.19991110,income,1,2026-06-30,93777000000.0,30951000000.0


,security_id,endpoint,report_type,report_period,status,missing_reason,revision_id,revision_sequence,raw_batch_id,usable_from,first_observed_at,availability_basis,evidence_ref,group_completeness
0,cnstock.600000.SH.19991110,income,1,2024-12-31,value,None,terminal:be290169fc86e93935a7cfd66d3215b01dc2a...,None,b_f903f7b83f9a40ad9e8d9a9790e34a3a,2026-09-28T15:06:17.810743+00:00,2026-09-28T15:06:17.810743+00:00,first_observed_at,None,None
1,cnstock.600000.SH.19991110,income,1,2025-03-31,value,None,terminal:74ad5117e58f679f11a4c2852a6ae0881587e...,None,b_a4945cfc079d459dbd7e06ad259defcd,2026-09-28T15:06:18.283748+00:00,2026-09-28T15:06:18.283748+00:00,first_observed_at,None,None
2,cnstock.600000.SH.19991110,income,1,2025-06-30,value,None,terminal:542c8508772d0cefbdfd7b82bd04bfaae49d2...,None,b_0a4c0b5c5deb48ef9525b756fe5571a6,2026-09-28T15:06:19.027179+00:00,2026-09-28T15:06:19.027179+00:00,first_observed_at,None,None
3,cnstock.600000.SH.19991110,income,1,2025-09-30,value,None,terminal:e22928a1a1240c4818855692246014d478767...,None,b_98ad9b3093324b14a3c7e604da35def7,2026-09-28T15:06:19.412095+00:00,2026-09-28T15:06:19.412095+00:00,first_observed_at,None,None
4,cnstock.600000.SH.19991110,income,1,2025-12-31,value,None,terminal:060cd8f2487503accb8b607a2bf4041ee7c0f...,None,b_1a7403b5c1064591ba46560a43e121d6,2026-09-28T15:06:20.459198+00:00,2026-09-28T15:06:20.459198+00:00,first_observed_at,None,None
5,cnstock.600000.SH.19991110,income,1,2026-03-31,value,None,terminal:623990f554cf16d06bcc6eaca605acb0ac272...,None,b_1cf2fbd0e3054f46a32c7db075736fa6,2026-09-28T15:06:21.037797+00:00,2026-09-28T15:06:21.037797+00:00,first_observed_at,None,None
6,cnstock.600000.SH.19991110,income,1,2026-06-30,value,None,terminal:3f7ec15b3818615c970c12e0668141c8d7811...,None,b_603ea7e454994dffbcb995a44c2d35bd,2026-09-28T15:06:22.091878+00:00,2026-09-28T15:06:22.091878+00:00,first_observed_at,None,None


原生报告键: ['security_id', 'endpoint', 'report_type', 'report_period']
Reader 先按 cutoff 选可见 revision，再按报告期和 endpoint 投影。


### 从累计收入得到单季和 TTM

收入表里的流量是年内累计值。2026 年上半年收入为 93,777,000,000 元，一季度为 46,573,000,000 元，所以二季度单季收入为 47,204,000,000 元。

`single_quarter` 用组成报告做差分，`ttm` 接出过去四季。本样本在较早报告期缺组件，TTM 留空；到 2025-12-31，四季已齐，TTM 为 173,964,000,000 元。资产负债表的期末存量保持原值，不走累计差分。


In [16]:
# The source field is year-to-date cumulative revenue; derive only after PIT selection.
unit = income_events.field_meta['total_revenue']['unit']
basis = income_events.field_meta['total_revenue']['basis']
quarter = single_quarter(income_events, field='total_revenue', endpoint='income',
    report_type='1', basis=basis, unit=unit)
trailing = ttm(income_events, field='total_revenue', endpoint='income',
    report_type='1', basis=basis, unit=unit)
display(quarter.frame)
display(trailing.frame)
assert quarter.context['derivation']['input_snapshot_id'] == FULL_SNAPSHOT
assert trailing.field_meta['total_revenue']['basis'] == 'trailing_twelve_months'
statuses = pd.DataFrame([{'period':p['report_period'], 'status':p['status'],
    'missing_reason':p['missing_reason'], 'components':len(p['components'])}
    for p in trailing.field_meta['total_revenue']['by_key']])
display(statuses)
print('缺季度时结果保持空值；单季和 TTM 均保存组成报告与固定版本。')


,security_id,endpoint,report_type,report_period,total_revenue
0,cnstock.600000.SH.19991110,income,1,2024-12-31,<NA>
1,cnstock.600000.SH.19991110,income,1,2025-03-31,45922000000.0
2,cnstock.600000.SH.19991110,income,1,2025-06-30,44637000000.0
3,cnstock.600000.SH.19991110,income,1,2025-09-30,41721000000.0
4,cnstock.600000.SH.19991110,income,1,2025-12-31,41684000000.0
5,cnstock.600000.SH.19991110,income,1,2026-03-31,46573000000.0
6,cnstock.600000.SH.19991110,income,1,2026-06-30,47204000000.0


,security_id,endpoint,report_type,report_period,total_revenue
0,cnstock.600000.SH.19991110,income,1,2024-12-31,<NA>
1,cnstock.600000.SH.19991110,income,1,2025-03-31,<NA>
2,cnstock.600000.SH.19991110,income,1,2025-06-30,<NA>
3,cnstock.600000.SH.19991110,income,1,2025-09-30,<NA>
4,cnstock.600000.SH.19991110,income,1,2025-12-31,173964000000.0
5,cnstock.600000.SH.19991110,income,1,2026-03-31,174615000000.0
6,cnstock.600000.SH.19991110,income,1,2026-06-30,177182000000.0


,period,status,missing_reason,components
0,2024-12-31,source_missing,missing_quarter,7
1,2025-03-31,source_missing,missing_quarter,7
2,2025-06-30,source_missing,missing_quarter,7
3,2025-09-30,source_missing,missing_quarter,7
4,2025-12-31,value,None,7
5,2026-03-31,value,None,7
6,2026-06-30,value,None,7


缺季度时结果保持空值；单季和 TTM 均保存组成报告与固定版本。


### 还需要现金流、资产或股东信息时

沿用事件查询，换域、字段和筛选条件即可。下面保留了资产、现金流、指标、分红和股东组的读取例子，可按需要展开。

这些结果仍处在各自的报告或事件键上。怎样选出每个研究日可用的那一份，接下来用披露与接收时间说明。


In [21]:
# Each event stays at its native economic key; no daily forward fill.
examples = (
    ('balance_sheet_events', ('total_assets','total_liabilities'), {'endpoint':'balancesheet','report_type':'1'}),
    ('cash_flow_events', ('operating_cash_flow',), {'endpoint':'cashflow','report_type':'1'}),
    ('financial_indicator_events', ('roe','debt_to_assets'), {'endpoint':'fina_indicator'}),
    ('corporate_actions', ('process_status','cash_dividend_before_tax_per_share','ex_date'), {}),
    ('top_holders_reports', ('holders','holder_count','group_completeness','top10_ratio'), {}),
)
event_results = {}

In [22]:
for domain_name, requested, filters in examples:
    segment = full_manifest['domains'][domain_name]
    rows = [r for p in segment['partitions']
            for r in full.store.read_partition(p).to_pylist()]
    if not rows:
        print(domain_name, '当前范围无事件；保留真实空结果')
        continue
    selected_id = bank_id if any(r['security_id'] == bank_id for r in rows) else rows[0]['security_id']
    rows = [r for r in rows if r['security_id'] == selected_id]
    date_field = 'report_period'
    dates = sorted(str(r[date_field]) for r in rows)
    fields = tuple(f for f in requested if f in segment['contract']['fields'] and f not in segment['contract']['logical_key'])
    if not fields:
        print(domain_name, '无可请求的非键字段')
        continue
    q = EventQuery(domain_name, fields, (selected_id,), dates[0], dates[-1], cutoff_now,
        'operational_pit_v1', date_field, filters)
    result = full.events(snapshot=FULL_SNAPSHOT, query=q)
    event_results[domain_name] = result
    print(domain_name, 'security:', selected_id, 'rows:', len(result.frame),
          'key:', segment['contract']['logical_key'])
    display(result.frame.head(12))
    print('limitation:', result.context.get('limitations'))

balance_sheet_events

 security: cnstock.600000.SH.19991110 rows: 6 key: ['security_id', 'endpoint', 'report_type', 'report_period']


,security_id,endpoint,report_type,report_period,total_assets,total_liabilities
0,cnstock.600000.SH.19991110,balancesheet,1,2024-12-31,9461880000000.0,8717099000000.0
1,cnstock.600000.SH.19991110,balancesheet,1,2025-03-31,9552276000000.0,8796647000000.0
2,cnstock.600000.SH.19991110,balancesheet,1,2025-06-30,9645791000000.0,8844539000000.0
3,cnstock.600000.SH.19991110,balancesheet,1,2025-09-30,9892214000000.0,9051370000000.0
4,cnstock.600000.SH.19991110,balancesheet,1,2025-12-31,10081746000000.0,9257316000000.0
5,cnstock.600000.SH.19991110,balancesheet,1,2026-03-31,10305646000000.0,9464247000000.0


limitation: ['report versions use declared supplier announcement days; same-day corrections use actual observation order, not a verified publication sequence']


cash_flow_events security: cnstock.600000.SH.19991110 rows: 4 key: ['security_id', 'endpoint', 'report_type', 'report_period']


...

,security_id,report_period,announcement_date,process_status,cash_dividend_before_tax_per_share,ex_date
0,cnstock.600000.SH.19991110,2024-12-31,2025-03-29,股东大会通过,0.41,None
1,cnstock.600000.SH.19991110,2024-12-31,2025-03-29,预案,0.41,None
2,cnstock.600000.SH.19991110,2024-12-31,2025-06-28,实施,0.41,2025-07-16
3,cnstock.600000.SH.19991110,2025-06-30,2025-08-28,股东大会通过,<NA>,None
4,cnstock.600000.SH.19991110,2025-06-30,2025-08-28,预案,0.0,None
5,cnstock.600000.SH.19991110,2025-12-31,2026-03-31,预案,0.42,None
6,cnstock.600000.SH.19991110,2025-12-31,2026-06-27,实施,0.42,2026-07-16
7,cnstock.600000.SH.19991110,2026-06-30,2026-08-28,股东大会通过,<NA>,None
8,cnstock.600000.SH.19991110,2026-06-30,2026-08-28,预案,0.0,None


limitation: ['terminal states ordered by system observation under declared source policy; vendor revision/publication order is unknown']


top_holders_reports security: cnstock.600000.SH.19991110 rows: 6 key: ['security_id', 'report_period']


,security_id,report_period,group_completeness,holders,holder_count,top10_ratio
0,cnstock.600000.SH.19991110,2025-03-31,supplier_report_complete,"[{""hold_amount_shares"":1395571025.0,""hold_rati...",10,76.2265
1,cnstock.600000.SH.19991110,2025-06-30,supplier_report_complete,"[{""hold_amount_shares"":1395571025.0,""hold_rati...",10,75.3584
2,cnstock.600000.SH.19991110,2025-09-30,supplier_report_complete,"[{""hold_amount_shares"":945568990.0,""hold_ratio...",10,74.6127
3,cnstock.600000.SH.19991110,2025-12-31,supplier_report_complete,"[{""hold_amount_shares"":945568990.0,""hold_ratio...",10,75.2567
4,cnstock.600000.SH.19991110,2026-03-31,supplier_report_complete,"[{""hold_amount_shares"":945568990.0,""hold_ratio...",10,74.9193
5,cnstock.600000.SH.19991110,2026-06-30,supplier_report_complete,"[{""hold_amount_shares"":945568990.0,""hold_ratio...",10,74.8702


limitation: ['report versions use declared supplier announcement days; same-day corrections use actual observation order, not a verified publication sequence']


In [23]:
actions = event_results.get('corporate_actions')
if actions is not None:
    statuses = actions.frame['process_status'].value_counts(dropna=False).rename_axis('process_status').reset_index(name='rows')
    display(statuses)
    implemented = actions.frame.loc[actions.frame['process_status'].eq('实施')]
    print('明确实施状态行:', len(implemented), '；其他状态不可直接当作已实施现金流。')
holders = event_results.get('top_holders_reports')
if holders is not None:
    display(holders.frame[['report_period','holder_count','group_completeness','top10_ratio']])
    assert all(pd.isna(value) for value in holders.frame.loc[
        ~holders.frame['group_completeness'].eq('supplier_report_complete'), 'top10_ratio'])
    print('supplier_report_complete 只表示当前来源响应满足十名不同持有人及字段检查；不是独立全量认证。')
print('价格涨跌停为当日未复权边界，不代表一定可成交。')
print('预测类数据不在这次 FullSourcePlan 中，不能把缺域解释为零预测。')


,process_status,rows
0,预案,4
1,股东大会通过,3
2,实施,2


明确实施状态行: 2 ；其他状态不可直接当作已实施现金流。


,report_period,holder_count,group_completeness,top10_ratio
0,2025-03-31,10,supplier_report_complete,76.2265
1,2025-06-30,10,supplier_report_complete,75.3584
2,2025-09-30,10,supplier_report_complete,74.6127
3,2025-12-31,10,supplier_report_complete,75.2567
4,2026-03-31,10,supplier_report_complete,74.9193
5,2026-06-30,10,supplier_report_complete,74.8702


supplier_report_complete 只表示当前来源响应满足十名不同持有人及字段检查；不是独立全量认证。
价格涨跌停为当日未复权边界，不代表一定可成交。
预测类数据不在这次 FullSourcePlan 中，不能把缺域解释为零预测。


## 5. 回到当时，能看见哪份报告

把财报放到历史价格旁边时，报告期只说明业绩归属，cutoff 才决定这一步允许知道什么。先看 `000001.SZ` 的一条分红：原公告日是 2026-05-23，实施公告日是 06-05，除息日是 06-12，每股税前现金为 0.36 元。

如果研究时点在 06-01，完整实施信息还未披露。将 cutoff 推到 06-08，按供应商公告日假设可读到实施行；按系统实际接收时间读取则仍为空，因为这份历史响应后来才收到。


In [17]:
# A real native implementation row has a later implementation announcement.
action_id = full_scope['identity_map']['000001.SZ']
def implemented_dividend_at(cutoff, policy):
    return full.events(snapshot=FULL_SNAPSHOT,
        query=EventQuery('corporate_actions',
            ('cash_dividend_before_tax_per_share',
             'implementation_announcement_date','record_date','ex_date'),
            (action_id,), '2026-05-23', '2026-05-23', cutoff, policy,
            'announcement_date', {'process_status':'实施'}))

early = implemented_dividend_at('2026-06-01T09:30:00+08:00',
    'best_effort_vendor_v1')
after_implementation = implemented_dividend_at('2026-06-08T09:30:00+08:00',
    'best_effort_vendor_v1')

In [18]:
strict_after = implemented_dividend_at('2026-06-08T09:30:00+08:00',
    'operational_pit_v1')
display(pd.DataFrame([
    {'cutoff':'2026-06-01 09:30','policy':'best_effort_vendor_v1',
     'visible_implementation_rows':len(early.frame)},
    {'cutoff':'2026-06-08 09:30','policy':'best_effort_vendor_v1',
     'visible_implementation_rows':len(after_implementation.frame)},
    {'cutoff':'2026-06-08 09:30','policy':'operational_pit_v1',
     'visible_implementation_rows':len(strict_after.frame)},
]))
assert len(early.frame) == 0
assert len(after_implementation.frame) == 1
assert len(strict_after.frame) == 0
implementation = after_implementation.frame.iloc[0]
assert str(implementation['implementation_announcement_date']) == '2026-06-05'

,cutoff,policy,visible_implementation_rows
0,2026-06-01 09:30,best_effort_vendor_v1,0
1,2026-06-08 09:30,best_effort_vendor_v1,1
2,2026-06-08 09:30,operational_pit_v1,0


In [19]:
assert str(implementation['record_date']) == '2026-06-11'
assert str(implementation['ex_date']) == '2026-06-12'
assert implementation['cash_dividend_before_tax_per_share'] == 0.36
display(after_implementation.frame)
print('原公告 05-23、实施公告 06-05；完整实施行不能在 06-01 预知。')
print('06-08 仅按供应商公告时间假设可见；严格 historical cutoff 仍早于原 Raw 接收。')

,security_id,report_period,announcement_date,process_status,cash_dividend_before_tax_per_share,implementation_announcement_date,record_date,ex_date
0,cnstock.000001.SZ.19910403,2025-12-31,2026-05-23,实施,0.36,2026-06-05,2026-06-11,2026-06-12


原公告 05-23、实施公告 06-05；完整实施行不能在 06-01 预知。
06-08 仅按供应商公告时间假设可见；严格 historical cutoff 仍早于原 Raw 接收。


### 选择历史信息口径

`operational_pit_v1` 按本系统实际收到时间读；`market_pit_safe_v1` 优先使用绑定具体修订的公开证据，缺证据时回到观察时间；`best_effort_vendor_v1` 使用已声明的供应商日期假设。

下面把这件事推到一个更容易看出差异的修订场景。

### 修订沿用同一公告日，会发生什么（合成反例）

一家公司 2024 年一季报归属 **03-31**，初版收入 **100 元**，公告日 **04-29**；配方声明下一开市日 **04-30 09:30** 才按供应商假设可用。05-08 发生修订，收入改为 **120 元**。这里故意假设供应商两版仍使用相同公告日 04-29，用来检验终态历史的限制。

- 及时日更：04-30 09:20 实收初版，05-09 09:20 实收修订。
- 晚到：初版直到 05-07 10:00 才收到，修订仍在 05-09 09:20 收到。
- 2026 回采：10-03 才收到终态 120，没有收到初版。

下表在各自包含全部上述接收记录的固定 Snapshot 中，重读历史 cutoff。时间均为北京时间；`空` 表示查询看不到任何该报告事件，事件查询返回零行。

| 研究 cutoff | 及时日更：按实际接收 | 晚到：按实际接收 | 2026 回采：按实际接收 | 按供应商公告日假设（这三个 Snapshot） |
|---|---:|---:|---:|---:|
| 04-30 09:35 | 100 | 空 | 空 | 120 |
| 05-07 09:35 | 100 | 空 | 空 | 120 |
| 05-07 10:05 | 100 | 100 | 空 | 120 |
| 05-09 09:35 | 120 | 120 | 空 | 120 |

“按实际接收”是 `operational_pit_v1`。本例没有修订绑定的精确公开时刻证据，因此 `market_pit_safe_v1` 的结果与它相同。“按供应商公告日假设”是 `best_effort_vendor_v1`：两版披露日相同，历史可见集合会选到终态 120，它不能重建初版 vintage。



本教程的历史研究沿用已声明的 best-effort 口径。若需要证明某个历史系统当时已经收到初版，仅有今天回采的终态记录不够，必须保留当时观察或对应版本的公开证据。

再回到 `600000.SH` 收入表，下面用 2026-06-01 09:30 比较三种口径。价格查询的同类比较保留在展开项里。


In [6]:
comparison = []
for policy in ('operational_pit_v1', 'market_pit_safe_v1', 'best_effort_vendor_v1'):
    result = DATA.read(snapshot=SNAPSHOT, query=query(policy=policy))
    comparison.append({'policy': policy, 'rows': len(result.frame),
                       'visible_close': int(result.frame.close.notna().sum()),
                       'limitation': '; '.join(result.context['limitations'])})
display(pd.DataFrame(comparison))
assert comparison[0]['visible_close'] == comparison[1]['visible_close'] == 0
assert comparison[2]['visible_close'] == len(DAYS)
print('一条真实字段来源：')
display(pd.DataFrame(batch.field_meta['close']['by_key'][:1]))


,policy,rows,visible_close,limitation
0,operational_pit_v1,6,0,terminal states ordered by system observation ...
1,market_pit_safe_v1,6,0,6 queried revisions lacked revision-bound publ...
2,best_effort_vendor_v1,6,6,"vendor release time is an assumption, not hist..."


一条真实字段来源：


,security_id,session,revision_id,revision_sequence,raw_batch_id,usable_from,first_observed_at,availability_basis,evidence_ref,missing_reason
0,cnstock.000001.SZ.19910403,2026-08-24,terminal:221697a48826b852e85b280f032a9368a2282...,None,b_71ba3777f9674756bbd6805dcebf4dd4,2026-08-24T20:00:00+08:00,2026-09-28T13:23:41.124858+00:00,declared_vendor_assumption,None,None


In [20]:
# Compare the same native report keys at a historical research clock.
historical_clock = '2026-06-01T09:30:00+08:00'
pit_rows = []
for policy in ('operational_pit_v1', 'market_pit_safe_v1', 'best_effort_vendor_v1'):
    selected = full.events(snapshot=FULL_SNAPSHOT,
        query=replace(income_query, cutoff=historical_clock, pit_policy=policy))
    pit_rows.append({'policy':policy, 'clock':historical_clock,
        'visible_reports':len(selected.frame),
        'limitations':'; '.join(selected.context.get('limitations', ()))})
display(pd.DataFrame(pit_rows))
assert pit_rows[0]['visible_reports'] == 0
assert pit_rows[1]['visible_reports'] == 0
print('今天收到的终态历史不会回填到 2026-06-01 的 operational/safe 信息集。')


,policy,clock,visible_reports,limitations
0,operational_pit_v1,2026-06-01T09:30:00+08:00,0,report versions use declared supplier announce...
1,market_pit_safe_v1,2026-06-01T09:30:00+08:00,0,7 event revisions lacked revision-bound public...
2,best_effort_vendor_v1,2026-06-01T09:30:00+08:00,6,vendor event availability is a declared assump...


今天收到的终态历史不会回填到 2026-06-01 的 operational/safe 信息集。


## 6. 选择当日候选，保留预热历史

现在价格和财报都能按研究时点读取，还要决定当日参考集合。成员查询采用截至 cutoff 最近可见的供应商整组快照，保留它的快照日期。

`000506.SZ` 在这里到 2026-06-30 才成为可见成员；06-01 继续使用之前的名单。先取足够的连续历史计算时序特征，再按当日成员选横截面，退出候选池也不应把已有历史窗口截断。


In [24]:
# A Tushare month-end snapshot is observed on its own trade_date.
MEMBER_SNAPSHOT = UNIFIED_SNAPSHOT
member_manifest = UNIFIED.store.load_snapshot(MEMBER_SNAPSHOT)
member_domain = member_manifest['domains']['universe_membership']
member_coverage = member_domain['coverage']
assert member_coverage['basis'] == 'tushare_dated_index_weight_snapshot_v1'
assert member_coverage['between_snapshots'] == 'carry_forward'
print('生产成员来源:', member_domain['source_profile']['id'])
print('固定 Snapshot:', MEMBER_SNAPSHOT,
      '；规范成员行:', sum(p['rows'] for p in member_domain['partitions']))
print('成员覆盖摘要:', {k:v for k,v in member_coverage.items()
    if k in ('basis','between_snapshots','universe_ids','observation_dates','limitations')})
state_rows = pd.DataFrame(member_coverage.get('complete_states', ()))

生产成员来源: tushare.index_weight.dated_membership.v1
固定 Snapshot: s_a0db6028118997d93b9f70dff852b701b030b152a30b2e8d325c08ad64931049 ；规范成员行: 3874
成员覆盖摘要: {'basis': 'tushare_dated_index_weight_snapshot_v1', 'between_snapshots': 'carry_forward', 'limitations': ['Supplier index_weight was observed on dated snapshots only; no intramonth membership change is proven.', 'Best-effort uses an assumed 18:00 Asia/Shanghai dated release; strict policies use actual Raw receipt.']}


In [25]:
if not state_rows.empty:
    selected = (state_rows.loc[state_rows.universe_id.eq('csi1800')]
        if 'universe_id' in state_rows else state_rows)
    display(selected)

MEMBER_DAYS = ('2026-07-31','2026-08-03','2026-08-31')
vendor_query = QuerySpec('universe_membership', ('is_member',), (SYMBOL,),
    MEMBER_DAYS, 'best_effort_vendor_v1',
    {day:day+'T20:30:00+08:00' for day in MEMBER_DAYS}, universe_id='csi1800')
vendor_members = UNIFIED.members(snapshot=MEMBER_SNAPSHOT, query=vendor_query)
display(vendor_members.frame)
member_meta = pd.DataFrame(vendor_members.field_meta['is_member']['by_key'])
meta_columns = [c for c in ('security_id','session','snapshot_date','source_snapshot_date',
    'effective_from','first_observed_at','availability_basis','missing_reason')
    if c in member_meta.columns]

,complete,dependency_raw_batch_ids,effective_from,effective_to,first_observed_at,member_count,member_set_sha256,member_set_source,raw_batch_id,revision_id,revision_sequence,source_snapshot_date,state_id,universe_id
12,True,"[b_53fad64807cc41448f028719847d2ee9, b_7277acc...",2026-05-29,2026-06-30,2026-09-28T13:17:24.908931+00:00,1800,4b9d1d17ccd9875474ec3f10369699179b5cdf2ef98b06...,canonical_intervals_v1,b_53fad64807cc41448f028719847d2ee9,vendor:9f0d23f6e5910baacf346b1379116ef805178b2...,1,2026-05-29,vendor:csi1800:2026-05-29,csi1800
13,True,"[b_b79a4a5d683548aaa11803d5edf648f3, b_1c3c1f6...",2026-06-30,2026-07-31,2026-09-28T13:17:26.380734+00:00,1800,869ca57470d399de6c11b1e3db3cde9d8b14fc705d1072...,canonical_intervals_v1,b_b79a4a5d683548aaa11803d5edf648f3,vendor:a8910eb6904cefcb0971b3a80d0349bd5cde133...,1,2026-06-30,vendor:csi1800:2026-06-30,csi1800
14,True,"[b_18fbd2b97d7440b8b8b17f96bdee23e5, b_4fe02a3...",2026-07-31,2026-08-31,2026-09-28T13:17:27.310742+00:00,1800,869ca57470d399de6c11b1e3db3cde9d8b14fc705d1072...,canonical_intervals_v1,b_18fbd2b97d7440b8b8b17f96bdee23e5,vendor:091f1143725c1fd73de240279684fd5e213a87c...,1,2026-07-31,vendor:csi1800:2026-07-31,csi1800
15,True,"[b_ccedf03a766d41c89d7e292efc276403, b_d2c26e2...",2026-08-31,None,2026-09-28T13:17:28.534869+00:00,1800,869ca57470d399de6c11b1e3db3cde9d8b14fc705d1072...,canonical_intervals_v1,b_ccedf03a766d41c89d7e292efc276403,vendor:1fe9cb9e4b4acc1c7443ff96c256dc5d26ed1c9...,1,2026-08-31,vendor:csi1800:2026-08-31,csi1800


,security_id,session,is_member
0,cnstock.000001.SZ.19910403,2026-07-31,True
1,cnstock.000001.SZ.19910403,2026-08-03,True
2,cnstock.000001.SZ.19910403,2026-08-31,True


In [26]:
display(member_meta[meta_columns])
snapshot_by_session = member_meta.set_index('session')['source_snapshot_date']
assert snapshot_by_session.loc['2026-08-03'] == '2026-07-31'
assert snapshot_by_session.loc['2026-08-31'] == '2026-08-31'
observed = vendor_members.frame.set_index('session').is_member
assert pd.notna(observed.loc['2026-07-31'])
assert observed.loc['2026-08-03'] == observed.loc['2026-07-31']
strict = UNIFIED.members(snapshot=MEMBER_SNAPSHOT,
    query=replace(vendor_query, pit_policy='operational_pit_v1'))
display(strict.frame)
assert strict.frame.is_member.isna().all()
print('08-03 沿用最近的 07-31 供应商快照；08-31 新快照不倒填 8 月初。')
print('这只是 Tushare 供应商研究口径；月内真实调样时刻没有逐日来源。')
CHANGED_ID = 'cnstock.000506.SZ.19930312'
CHANGE_DAYS = ('2026-05-29','2026-06-01','2026-06-30','2026-07-01')

,security_id,session,source_snapshot_date,first_observed_at,availability_basis,missing_reason
0,cnstock.000001.SZ.19910403,2026-07-31,2026-07-31,2026-09-28T13:17:24.908931+00:00,declared_vendor_assumption,None
1,cnstock.000001.SZ.19910403,2026-08-03,2026-07-31,2026-09-28T13:17:24.908931+00:00,declared_vendor_assumption,None
2,cnstock.000001.SZ.19910403,2026-08-31,2026-08-31,2026-09-28T13:17:24.908931+00:00,declared_vendor_assumption,None


,security_id,session,is_member
0,cnstock.000001.SZ.19910403,2026-07-31,<NA>
1,cnstock.000001.SZ.19910403,2026-08-03,<NA>
2,cnstock.000001.SZ.19910403,2026-08-31,<NA>


08-03 沿用最近的 07-31 供应商快照；08-31 新快照不倒填 8 月初。
这只是 Tushare 供应商研究口径；月内真实调样时刻没有逐日来源。


In [27]:
change_query = QuerySpec('universe_membership', ('is_member',), (CHANGED_ID,),
    CHANGE_DAYS, 'best_effort_vendor_v1',
    {day:day+'T20:30:00+08:00' for day in CHANGE_DAYS}, universe_id='csi1800')
changed = UNIFIED.members(snapshot=MEMBER_SNAPSHOT, query=change_query)
display(changed.frame)
assert changed.frame.is_member.tolist() == [False,False,True,True]
change_meta = pd.DataFrame(changed.field_meta['is_member']['by_key'])
display(change_meta[['session','source_snapshot_date','first_observed_at']])
assert change_meta.source_snapshot_date.tolist() == [
    '2026-05-29','2026-05-29','2026-06-30','2026-06-30']
print('000506 仅在供应商 06-30 新快照转为成员；没有倒填为 06-15 的精确调整。')

,security_id,session,is_member
0,cnstock.000506.SZ.19930312,2026-05-29,False
1,cnstock.000506.SZ.19930312,2026-06-01,False
2,cnstock.000506.SZ.19930312,2026-06-30,True
3,cnstock.000506.SZ.19930312,2026-07-01,True


,session,source_snapshot_date,first_observed_at
0,2026-05-29,2026-05-29,2026-09-28T13:17:24.908931+00:00
1,2026-06-01,2026-05-29,2026-09-28T13:17:24.908931+00:00
2,2026-06-30,2026-06-30,2026-09-28T13:17:26.380734+00:00
3,2026-07-01,2026-06-30,2026-09-28T13:17:26.380734+00:00


000506 仅在供应商 06-30 新快照转为成员；没有倒填为 06-15 的精确调整。


In [28]:
# This is a saved live Tushare probe, independent of the tutorial's production Snapshot.
cadence = json.loads((REPO / 'docs/tushare-membership-cadence.json').read_text())
probe_rows = []
for item in cadence['results']:
    params = item['params']
    probe_rows.append({'index_code':params['index_code'],
        'requested_from':params['start_date'], 'requested_to':params['end_date'],
        'rows':item['rows'], 'returned_trade_dates':','.join(item['trade_date_counts']),
        'raw_batch_id':item['raw_batch_id'], 'observed_at':item['observed_at']})
probe = pd.DataFrame(probe_rows)
display(probe)
assert len(probe) == 12
for index_code, expected in (('000300.SH',300),('000905.SH',500),('000852.SH',1000)):
    segment = probe.loc[probe.index_code.eq(index_code)]
    assert segment.loc[segment.requested_from.isin(('20260612','20260615')), 'rows'].eq(0).all()
    assert segment.loc[segment.requested_from.eq('20260630'), 'rows'].iloc[0] == expected
    assert segment.loc[segment.requested_from.eq('20260601'), 'returned_trade_dates'].iloc[0] == '20260630'
print('实际来源仅在六月末返回同日名单；逐日请求不能制造逐日事实。')
MONTH_ENDS = ('2026-07-31', '2026-08-31')


,index_code,requested_from,requested_to,rows,returned_trade_dates,raw_batch_id,observed_at
0,000300.SH,20260612,20260612,0,,b_b382f8701049473c9645468637c02f24,2026-10-03T05:02:05.724878+00:00
1,000300.SH,20260615,20260615,0,,b_91cc3b1bfd944b989ce71f08c663650a,2026-10-03T05:02:06.109068+00:00
2,000300.SH,20260630,20260630,300,20260630,b_862fbc2c4a41440c8598c7d9aeeae859,2026-10-03T05:02:06.507870+00:00
3,000300.SH,20260601,20260630,300,20260630,b_14e33c6e9be941bcb0408e2def4813ab,2026-10-03T05:02:06.884449+00:00
4,000905.SH,20260612,20260612,0,,b_bce58844823b4a78b950182079d2d911,2026-10-03T05:02:07.277555+00:00
5,000905.SH,20260615,20260615,0,,b_bf0f69de259d4d5f91afb1fad56d0063,2026-10-03T05:02:07.652712+00:00
6,000905.SH,20260630,20260630,500,20260630,b_9613b19b682d47aeaadd3f04ed526ad3,2026-10-03T05:02:08.045632+00:00
7,000905.SH,20260601,20260630,500,20260630,b_3bb98aaff5884434810c8de3fcec7bad,2026-10-03T05:02:08.437697+00:00
8,000852.SH,20260612,20260612,0,,b_d2386f37ddbb4d78a5a5248c136072b8,2026-10-03T05:02:08.886354+00:00
9,000852.SH,20260615,20260615,0,,b_bd34b0aafbd248a398cf57325ff24ae0,2026-10-03T05:02:09.245978+00:00


实际来源仅在六月末返回同日名单；逐日请求不能制造逐日事实。


## 7. 构建特征，再重复一次实验

先把价格查询保存下来。这里记录固定 Snapshot、QuerySpec 与 Reader 版本，换一个 Data 实例后按这些参数重读，表格与来源相等。

这一步保存的是输入查询。接下来把价格和财报组成 FeatureBuild，才有可以直接复用的特征结果。


In [29]:
saved_query = {'snapshot_id': SNAPSHOT, 'domain': batch.context['domain'],
               'query': batch.context['query'],
               'reader_version': batch.context['reader_version']}
view_path = TEMP / 'query.json'
view_path.write_text(json.dumps(saved_query, ensure_ascii=False))
reloaded = json.loads(view_path.read_text())
replay_query = dict(reloaded['query'])
for key in ('fields', 'symbols', 'sessions'):
    replay_query[key] = tuple(replay_query[key])
replayed = Data(ROOT).read(snapshot=reloaded['snapshot_id'], query=QuerySpec(domain=reloaded['domain'], **replay_query))
assert replayed.context['reader_version'] == reloaded['reader_version']
assert replayed.to_json() == batch.to_json()
print('查询 JSON bytes:', view_path.stat().st_size, '重放相等:', True)
measurements = []
cached_data = Data(ROOT)
for label in ('冷Reader（不清操作系统缓存）', '暖Reader（同一查询）'):
    reader = cached_data._reader(SNAPSHOT)
    before = reader.partition_reads
    started = time.perf_counter()
    measured = cached_data.read(snapshot=SNAPSHOT, query=query())
    measurements.append({'读取': label, '秒': round(time.perf_counter()-started, 5),
                         '新增分区读取': reader.partition_reads-before, '累计查询缓存命中': reader.cache_hits})
    assert measured.to_json() == DATA.read(snapshot=SNAPSHOT, query=query()).to_json()
display(pd.DataFrame(measurements))
assert measurements[0]['新增分区读取'] >= 1 and measurements[1]['新增分区读取'] == 0
print('Reader 暖读取复用进程内缓存；持久复用的 Qlib 在后文示范。此处没有实际 FeatureBuild 存储。')

查询 JSON bytes: 799 重放相等: True


,读取,秒,新增分区读取,累计查询缓存命中
0,冷Reader（不清操作系统缓存）,0.00400,1,0
1,暖Reader（同一查询）,0.00015,0,1


Reader 暖读取复用进程内缓存；持久复用的 Qlib 在后文示范。此处没有实际 FeatureBuild 存储。


### 把行情和当时可见的财报对齐

回到两证券样本，将 `000001.SZ` 和 `600000.SH` 的六个交易日组成面板。两条行情查询读取 close/volume 与每日限价，三条财务查询读取收入、资产和 ROE。每一天先由 Data 按它自己的 cutoff 选修订，再为每条财务流选择当时可见的最新报告期。

查询用 `daily`、`limits`、`income`、`balance`、`indicator` 命名，面板列也带相同前缀。这六天里收入流实际换了报告期，可以展开选择记录看切换落在哪个 cutoff。

Research 编排这些输入，Core 执行 identity、pct_change 和 asof。这里的原价 return 用于说明算子；用于价格收益的正式配方应接上第 3 节的价格口径。


In [2]:
sys.path.insert(0, str(WORKSPACE / 'axiom-engine' / 'src'))
from axiom_research import ArtifactRef, build_joint_features, load_feature_build
import axiom_research, subprocess

# Actual retained runtime packages; Core's source is the unchanged reviewed commit.
runtime_binding = {'research_version':'0.1.1',
    'research_wheel_sha256':'47ab757918003e372cb89ee59e73ea261debdc697d6ae33d9e6ac075bdedf696',
    'data_version':'0.3.5',
    'data_wheel_sha256':'1773c1e588b1802d976c7766e80854060875e767bbb869bd5eae93bdae9bb75f',
    'core_commit':'c8a506b2070a9c89b20cfe14a0a93155c238f204'}
binding_bytes = json.dumps(runtime_binding, sort_keys=True, separators=(',', ':')).encode()
implementation = ArtifactRef(artifact_type='ImplementationPackage',
    artifact_id='joint-research-0.1.1-runtime', artifact_contract_version='1',
    content_digest='sha256:' + hashlib.sha256(binding_bytes).hexdigest(),
    uri='inline:reviewed-runtime', metadata={'runtime_binding':runtime_binding})
scope = json.loads((REPO / 'delivery/csi1800_tushare_only_v4_202606_202608.scope.json').read_text())
symbols = tuple(scope['identity_map'][code] for code in ('000001.SZ','600000.SH'))
joint_days = ('2026-08-24','2026-08-25','2026-08-26','2026-08-27','2026-08-28','2026-08-31')
cutoffs = {day:day+'T20:30:00+08:00' for day in joint_days}
market_queries = {
    'daily': QuerySpec('market_daily', ('close','volume_shares'), symbols, joint_days,
        'best_effort_vendor_v1', cutoffs),
    'limits': QuerySpec('price_limits', ('up_limit','down_limit'), symbols, joint_days,
        'best_effort_vendor_v1', cutoffs)}
members = QuerySpec('universe_membership', ('is_member',), symbols, joint_days,
    'best_effort_vendor_v1', cutoffs, universe_id='csi1800')
financial_queries = {
    'income': EventQuery('financial_events', ('total_revenue','parent_net_income'), symbols,
        '2020-01-01','2026-08-31',cutoffs[joint_days[-1]],'best_effort_vendor_v1',
        'report_period', {'endpoint':'income','report_type':'1'}),
    'balance': EventQuery('balance_sheet_events', ('total_assets',), symbols,
        '2020-01-01','2026-08-31',cutoffs[joint_days[-1]],'best_effort_vendor_v1',
        'report_period', {'endpoint':'balancesheet','report_type':'1'}),
    'indicator': EventQuery('financial_indicator_events', ('roe',), symbols,
        '2020-01-01','2026-08-31',cutoffs[joint_days[-1]],'best_effort_vendor_v1',
        'report_period', {'endpoint':'fina_indicator'})}
params = dict(snapshot=UNIFIED_SNAPSHOT, market_queries=market_queries,
    membership_query=members, financial_queries=financial_queries,
    destination=TEMP/'joint-builds', implementation_package=implementation)
started=time.perf_counter()
built=build_joint_features(UNIFIED, **params)
build_seconds=time.perf_counter()-started
panel=built.read()
display(panel)
saved=built.evidence()
selected=[]
for entry in saved['source_evidence'].values():
    if 'report_selection' not in entry: continue
    ctx=saved['query_contexts'][entry['query_context_ref']]
    meta=entry['provenance']
    selected.append({'输入':entry['alias'],'字段':entry['field'], '证券':meta['security_id'],
        '决策cutoff':ctx['query']['cutoff'],'报告期':meta['report_period'],
        'revision':meta['revision_id'],'首次收到':meta['first_observed_at'],
        '可用时间':meta['usable_from'],'缺失原因':meta['missing_reason'],
        '时间依据':meta['availability_basis']})
display(pd.DataFrame(selected))
assert len({r['报告期'] for r in selected if r['输入']=='income'})>1
print('收入流在这六个 session 内实际切换了报告期，仍按各自 cutoff 选择。')
assert len(panel)==len(symbols)*len(joint_days) and len(panel.columns)==14
assert panel['income__total_revenue'].notna().any()
mtimes={p.name:p.stat().st_mtime_ns for p in built.path.iterdir()}
started=time.perf_counter()
reused=build_joint_features(Data(UNIFIED_ROOT), **params)
reuse_seconds=time.perf_counter()-started
assert reused.reused and reused.read().equals(panel)
assert {p.name:p.stat().st_mtime_ns for p in built.path.iterdir()}==mtimes
# An independent process opens the persisted build without any Data root.
child_code="from axiom_research import load_feature_build; import sys,json,hashlib; p=load_feature_build(sys.argv[1]).read(); print(json.dumps({'rows':len(p),'columns':len(p.columns),'panel_sha256':hashlib.sha256(p.to_json().encode()).hexdigest()}))"
child_env={**os.environ,'PYTHONPATH':os.pathsep.join((str(Path(axiom_research.__file__).parents[1]),
    str(WORKSPACE/'axiom-engine/src'),PACKAGE_PATH or str(WORKSPACE/'axiom-data/src')))}
child=json.loads(subprocess.check_output([sys.executable,'-c',child_code,str(built.path)],env=child_env,text=True))
assert child['panel_sha256']==hashlib.sha256(panel.to_json().encode()).hexdigest()
assert (UNIFIED_ROOT/'current.json').read_bytes()==UNIFIED_CURRENT_BEFORE
display(pd.DataFrame([{'证券':len(symbols),'sessions':len(joint_days),'行情查询':len(market_queries),
    '财务查询':len(financial_queries),'输出行':len(panel),'输出列':len(panel.columns)-2,
    '构建秒':round(build_seconds,4),'复用秒':round(reuse_seconds,4),'跨进程重读':True,
    '供应商请求':0,'正式数据根写入':False}]))
print('本例是既存真实 Data 的 best-effort 消费；不是严格历史公开证据或多年性能验收。')


,security_id,session,daily__close,daily__close__return,daily__volume_shares,daily__volume_shares__return,limits__up_limit,limits__up_limit__return,limits__down_limit,limits__down_limit__return,income__total_revenue,income__parent_net_income,balance__total_assets,indicator__roe
0,cnstock.000001.SZ.19910403,2026-08-24,11.56,NaN,119902495.0,NaN,12.55,NaN,10.27,NaN,7.061700e+10,2.569600e+10,6.028785e+12,4.6746
1,cnstock.600000.SH.19991110,2026-08-24,9.22,NaN,79110605.0,NaN,9.96,NaN,8.15,NaN,4.657300e+10,1.786100e+10,1.030565e+13,2.1641
2,cnstock.000001.SZ.19910403,2026-08-25,11.59,0.002595,99488115.0,-0.170258,12.72,0.013546,10.40,0.012658,7.061700e+10,2.569600e+10,6.028785e+12,4.6746
3,cnstock.600000.SH.19991110,2026-08-25,9.08,-0.015184,88175624.0,0.114587,10.14,0.018072,8.30,0.018405,4.657300e+10,1.786100e+10,1.030565e+13,2.1641
4,cnstock.000001.SZ.19910403,2026-08-26,11.73,0.012079,116728878.0,0.173295,12.75,0.002358,10.43,0.002885,7.061700e+10,2.569600e+10,6.028785e+12,4.6746
5,cnstock.600000.SH.19991110,2026-08-26,9.21,0.014317,95070325.0,0.078193,9.99,-0.014793,8.17,-0.015663,4.657300e+10,1.786100e+10,1.030565e+13,2.1641
6,cnstock.000001.SZ.19910403,2026-08-27,11.59,-0.011935,97570170.0,-0.164130,12.90,0.011765,10.56,0.012464,7.061700e+10,2.569600e+10,6.028785e+12,4.6746
7,cnstock.600000.SH.19991110,2026-08-27,9.07,-0.015201,95810090.0,0.007781,10.13,0.014014,8.29,0.014688,4.657300e+10,1.786100e+10,1.030565e+13,2.1641
8,cnstock.000001.SZ.19910403,2026-08-28,11.65,0.005177,83812624.0,-0.141002,12.75,-0.011628,10.43,-0.012311,7.061700e+10,2.569600e+10,6.028785e+12,4.6746
9,cnstock.600000.SH.19991110,2026-08-28,9.00,-0.007718,58786810.0,-0.386424,9.98,-0.014808,8.16,-0.015682,4.657300e+10,1.786100e+10,1.030565e+13,2.1641


,输入,字段,证券,决策cutoff,报告期,revision,首次收到,可用时间,缺失原因,时间依据
0,indicator,roe,cnstock.000001.SZ.19910403,2026-08-24T20:30:00+08:00,2026-06-30,terminal:4759e6dc5533537dab7fb5d7254c02c9e0887...,2026-09-28T15:06:37.140541+00:00,2026-08-17T09:30:00+08:00,None,declared_vendor_assumption
1,income,total_revenue,cnstock.000001.SZ.19910403,2026-08-26T20:30:00+08:00,2026-06-30,terminal:5604a10f710966c0e382c611893800d5cf341...,2026-09-28T15:06:22.091878+00:00,2026-08-17T09:30:00+08:00,None,declared_vendor_assumption
2,indicator,roe,cnstock.000001.SZ.19910403,2026-08-26T20:30:00+08:00,2026-06-30,terminal:4759e6dc5533537dab7fb5d7254c02c9e0887...,2026-09-28T15:06:37.140541+00:00,2026-08-17T09:30:00+08:00,None,declared_vendor_assumption
3,indicator,roe,cnstock.600000.SH.19991110,2026-08-28T20:30:00+08:00,2026-03-31,terminal:63066a23fa3a83710a19474e5d7d9901d52f3...,2026-09-28T15:06:36.854781+00:00,2026-05-06T09:30:00+08:00,None,declared_vendor_assumption
4,balance,total_assets,cnstock.000001.SZ.19910403,2026-08-26T20:30:00+08:00,2026-06-30,terminal:28bfc3da331e968afbf95b9e59964d45396a0...,2026-09-28T15:06:27.616936+00:00,2026-08-17T09:30:00+08:00,None,declared_vendor_assumption
5,income,total_revenue,cnstock.600000.SH.19991110,2026-08-25T20:30:00+08:00,2026-03-31,terminal:623990f554cf16d06bcc6eaca605acb0ac272...,2026-09-28T15:06:21.037797+00:00,2026-05-06T09:30:00+08:00,None,declared_vendor_assumption
6,balance,total_assets,cnstock.000001.SZ.19910403,2026-08-31T20:30:00+08:00,2026-06-30,terminal:28bfc3da331e968afbf95b9e59964d45396a0...,2026-09-28T15:06:27.616936+00:00,2026-08-17T09:30:00+08:00,None,declared_vendor_assumption
7,indicator,roe,cnstock.600000.SH.19991110,2026-08-24T20:30:00+08:00,2026-03-31,terminal:63066a23fa3a83710a19474e5d7d9901d52f3...,2026-09-28T15:06:36.854781+00:00,2026-05-06T09:30:00+08:00,None,declared_vendor_assumption
8,indicator,roe,cnstock.600000.SH.19991110,2026-08-25T20:30:00+08:00,2026-03-31,terminal:63066a23fa3a83710a19474e5d7d9901d52f3...,2026-09-28T15:06:36.854781+00:00,2026-05-06T09:30:00+08:00,None,declared_vendor_assumption
9,income,parent_net_income,cnstock.000001.SZ.19910403,2026-08-31T20:30:00+08:00,2026-06-30,terminal:5604a10f710966c0e382c611893800d5cf341...,2026-09-28T15:06:22.091878+00:00,2026-08-17T09:30:00+08:00,None,declared_vendor_assumption


收入流在这六个 session 内实际切换了报告期，仍按各自 cutoff 选择。


,证券,sessions,行情查询,财务查询,输出行,输出列,构建秒,复用秒,跨进程重读,供应商请求,正式数据根写入
0,2,6,2,3,12,12,0.6836,0.1773,True,0,False


本例是既存真实 Data 的 best-effort 消费；不是严格历史公开证据或多年性能验收。


再次调用同样的构建参数，Research 复用已经保存的 FeatureBuild。这次两证券、六天面板约 0.68 秒构建、0.18 秒复用。换一个 Data 实例，面板和文件 mtime 也保持一致；另一 Python 进程可以直接加载产物，无需重新访问 Data。

因此，后续换一组下游实验参数，可以先复用这张面板。改了输入字段、时间口径或特征配方，再建立对应的新 build。


## 8. 把固定输入交给下游

将这套读法用到七只 ETF 时，身份、数量单位和数据源换成基金合同。已有样本覆盖 2026-06—08 月，并从 03-18 保留预热，合计 114 个交易日。

以 08-31 的周首个交易日为例：决策读取 08-28 的价格与因子，执行侧另读 08-31 的开盘价和价格限制。当天完整日线只在回放用途下使用。下面把这两次读取摆在一起。


In [30]:
# A second fixed, real Tushare Snapshot holds only the seven ETF data scope.
ETF_ROOT = Path(os.environ.get('AXIOM_ETF_DATA_ROOT',
    WORKSPACE / 'data/etf_rotation_daily_202606_202608'))
ETF_SNAPSHOT = os.environ.get('AXIOM_ETF_SNAPSHOT',
    's_e32f4511b69d4fce9accad609feb4005f371cc5cb833ca260d6ae563b75558d1')
ETF = Data(ETF_ROOT)
ETF_CURRENT_BEFORE = (ETF_ROOT/'current.json').read_bytes()
ETF_SCOPE = json.loads(Path(os.environ.get('AXIOM_ETF_SCOPE',
    REPO / 'delivery/etf_rotation_daily_202606_202608.scope.json')).read_text())
ETF_REPORT = json.loads(Path(os.environ.get('AXIOM_ETF_REPORT',
    REPO / 'delivery/etf_rotation_daily_202606_202608.report.json')).read_text())
assert ETF.resolve('current') == ETF_SNAPSHOT
assert ETF_REPORT['snapshot_id'] == ETF_SNAPSHOT
assert ETF_REPORT['verify']['verified'] and ETF_REPORT['audit']['status'] == 'passed'
etf_manifest = ETF.store.load_snapshot(ETF_SNAPSHOT)

In [31]:
etf_ids = tuple(ETF_SCOPE['identity_map'][symbol] for symbol in ETF_SCOPE['symbols'])
master = [r for p in etf_manifest['domains']['security_master']['partitions']
          for r in ETF.store.read_partition(p).to_pylist()]
assert {r['security_id'] for r in master} == set(etf_ids)
display(pd.DataFrame(master)[['source_code','security_id','name','asset_type','listing_date','list_status']])
calendar = [r for p in etf_manifest['domains']['trading_calendar']['partitions']
            for r in ETF.store.read_partition(p).to_pylist()]
open_sse = sorted(str(r['session']) for r in calendar if r['exchange'] == 'SSE'
                  and r['is_open'] and str(r['session']) <= ETF_SCOPE['end_session'])
print('ETF scope:', ETF_SCOPE['start_session'], '—', ETF_SCOPE['end_session'],
      '；预热从', ETF_SCOPE['warmup_start_session'], '开始；日线实际交易日:',
      len(open_sse), '；首尾:', open_sse[0], open_sse[-1])
assert sum(p['rows'] for p in etf_manifest['domains']['market_daily']['partitions']) == 7*114
assert sum(p['rows'] for p in etf_manifest['domains']['corporate_actions']['partitions']) == 44

,source_code,security_id,name,asset_type,listing_date,list_status
0,510880.SH,cn.etf.SSE.510880.20070118,华泰柏瑞上证红利ETF,exchange_traded_fund,2007-01-18,L
1,510300.SH,cn.etf.SSE.510300.20120528,华泰柏瑞沪深300ETF,exchange_traded_fund,2012-05-28,L
2,510500.SH,cn.etf.SSE.510500.20130315,南方中证500ETF,exchange_traded_fund,2013-03-15,L
3,511010.SH,cn.etf.SSE.511010.20130325,国泰上证5年期国债ETF,exchange_traded_fund,2013-03-25,L
4,513100.SH,cn.etf.SSE.513100.20130515,国泰纳斯达克100ETF(QDII),exchange_traded_fund,2013-05-15,L
5,518880.SH,cn.etf.SSE.518880.20130729,华安易富黄金ETF,exchange_traded_fund,2013-07-29,L
6,159915.SZ,cn.etf.SZSE.159915.20111209,易方达创业板ETF,exchange_traded_fund,2011-12-09,L


ETF scope: 2026-06-01 — 2026-08-31 ；预热从 2026-03-18 开始；日线实际交易日: 114 ；首尾: 2026-03-18 2026-08-31


In [32]:
def etf_read(domain, fields, day, cutoff, *, purpose='decision_facts', securities=etf_ids):
    return ETF.read(snapshot=ETF_SNAPSHOT, query=QuerySpec(domain, tuple(fields),
        tuple(securities), (day,), 'best_effort_vendor_v1', {day:cutoff}, purpose=purpose))

prior_day = '2026-08-28'
current_day = '2026-08-31'
prior_cutoff = prior_day+'T20:30:00+08:00'
prior_bar = etf_read('market_daily',
    ('open','high','low','close','volume_units','amount_cny'), prior_day, prior_cutoff)
prior_factor = etf_read('adjustment_factors', ('factor',), prior_day, prior_cutoff)
benchmark = etf_read('benchmark_daily', ('close',), prior_day, prior_cutoff,
    securities=('000300.SH',))
limits = etf_read('price_limits', ('up_limit','down_limit'), current_day,
    current_day+'T09:00:00+08:00')
# An opening price is a historical market-replay outcome. It is never a same-day
# input to a pre-open decision, and the rest of that day's bar is not queried here.

In [33]:
replay_open = etf_read('market_daily', ('open',), current_day,
    current_day+'T20:30:00+08:00', purpose='market_replay')
assert all(len(batch.frame) == 7 for batch in
           (prior_bar, prior_factor, limits, replay_open))
assert len(benchmark.frame) == 1
display(prior_bar.frame)
display(prior_factor.frame)
display(limits.frame)
display(benchmark.frame)
display(replay_open.frame)
print('单位:', {k:v['unit'] for k,v in prior_bar.field_meta.items()})
print('日线一条真实来源:', prior_bar.field_meta['close']['by_key'][0])
strict_old = ETF.read(snapshot=ETF_SNAPSHOT,
    query=QuerySpec('market_daily', ('close',), etf_ids, (prior_day,),
        'operational_pit_v1', {prior_day:prior_cutoff}))

,security_id,session,open,high,low,close,volume_units,amount_cny
0,cn.etf.SZSE.159915.20111209,2026-08-28,3.478,3.539,3.44,3.443,1485427127,5179583482.6
1,cn.etf.SSE.510300.20120528,2026-08-28,4.684,4.706,4.675,4.679,604693322,2835120646.0
2,cn.etf.SSE.510500.20130315,2026-08-28,7.96,8.004,7.913,7.923,267221620,2125117295.0
3,cn.etf.SSE.510880.20070118,2026-08-28,3.45,3.475,3.425,3.461,132670400,457855094.0
4,cn.etf.SSE.511010.20130325,2026-08-28,141.137,141.137,141.08,141.112,18388500,2594813442.0
5,cn.etf.SSE.513100.20130515,2026-08-28,2.244,2.247,2.229,2.235,108671134,243076831.0
6,cn.etf.SSE.518880.20130729,2026-08-28,9.437,9.475,9.398,9.475,473701359,4462695029.0


,security_id,session,factor
0,cn.etf.SZSE.159915.20111209,2026-08-28,1.0
1,cn.etf.SSE.510300.20120528,2026-08-28,1.2671
2,cn.etf.SSE.510500.20130315,2026-08-28,0.3401
3,cn.etf.SSE.510880.20070118,2026-08-28,1.8136
4,cn.etf.SSE.511010.20130325,2026-08-28,1.0375
5,cn.etf.SSE.513100.20130515,2026-08-28,5.0019
6,cn.etf.SSE.518880.20130729,2026-08-28,1.0


,security_id,session,up_limit,down_limit
0,cn.etf.SZSE.159915.20111209,2026-08-31,4.132,2.754
1,cn.etf.SSE.510300.20120528,2026-08-31,5.147,4.211
2,cn.etf.SSE.510500.20130315,2026-08-31,8.715,7.131
3,cn.etf.SSE.510880.20070118,2026-08-31,3.807,3.115
4,cn.etf.SSE.511010.20130325,2026-08-31,155.223,127.001
5,cn.etf.SSE.513100.20130515,2026-08-31,2.459,2.012
6,cn.etf.SSE.518880.20130729,2026-08-31,10.423,8.528


,security_id,session,close
0,000300.SH,2026-08-28,4609.1764


,security_id,session,open
0,cn.etf.SZSE.159915.20111209,2026-08-31,3.389
1,cn.etf.SSE.510300.20120528,2026-08-31,4.64
2,cn.etf.SSE.510500.20130315,2026-08-31,7.85
3,cn.etf.SSE.510880.20070118,2026-08-31,3.466
4,cn.etf.SSE.511010.20130325,2026-08-31,141.116
5,cn.etf.SSE.513100.20130515,2026-08-31,2.207
6,cn.etf.SSE.518880.20130729,2026-08-31,9.16


单位: {'open': 'CNY/fund unit', 'high': 'CNY/fund unit', 'low': 'CNY/fund unit', 'close': 'CNY/fund unit', 'volume_units': 'fund units', 'amount_cny': 'CNY'}
日线一条真实来源: {'security_id': 'cn.etf.SZSE.159915.20111209', 'session': '2026-08-28', 'revision_id': 'terminal:107290163259fbe5d5eff6c8c2a8b5d2df16e08f0b050fdb33333a56d575c949', 'revision_sequence': None, 'raw_batch_id': 'b_6a5de10d5b84462199a50c3970d8b133', 'usable_from': '2026-08-28T20:00:00+08:00', 'first_observed_at': '2026-10-03T06:01:34.876904+00:00', 'availability_basis': 'declared_vendor_assumption', 'evidence_ref': None, 'missing_reason': None}


In [34]:
assert strict_old.frame['close'].isna().all()

dividends = ETF.events(snapshot=ETF_SNAPSHOT,
    query=EventQuery('corporate_actions',
        ('cash_dividend_per_unit','ex_date','pay_date'), etf_ids,
        ETF_SCOPE['warmup_start_session'], ETF_SCOPE['end_session'],
        datetime.now(timezone.utc).isoformat(), 'operational_pit_v1',
        'announcement_date', {}))
display(dividends.frame)
print('分红原生键:', etf_manifest['domains']['corporate_actions']['contract']['logical_key'])
print('当前窗口内可见事件:', len(dividends.frame),
      '；全 Snapshot 来源事件行:',
      sum(p['rows'] for p in etf_manifest['domains']['corporate_actions']['partitions']))
print('旧日严格 cutoff 早于原 Raw 接收，不能回填日线可见性。')

,security_id,announcement_date,process_status,cash_dividend_per_unit,ex_date,pay_date
0,cn.etf.SSE.510500.20130315,2026-07-08,实施,0.149,2026-07-15,2026-07-20
1,cn.etf.SSE.511010.20130325,2026-03-20,实施,0.5715,2026-03-25,2026-03-30
2,cn.etf.SSE.511010.20130325,2026-06-22,实施,0.8144,2026-06-25,2026-06-30


分红原生键: ['security_id', 'announcement_date', 'process_status']
当前窗口内可见事件: 3 ；全 Snapshot 来源事件行: 44
旧日严格 cutoff 早于原 Raw 接收，不能回填日线可见性。


这份固定 ETF 数据已经用于 Research 的 20D 信号实验，并由 Engine 消费最终信号与独立 market replay。模拟账户保存的意图、成交、费用、现金分红和净值可以交给只读报告查看，具体入口见[当前交付](../docs/current-delivery.md)。

从这里开始，换组合或执行参数影响下游回测；前面已经固定的事实与信号仍可复用。


## 9. 改特征或修正数据后，旧实验怎么办

现在输入查询和 FeatureBuild 都已经保存。若只想把 20D 窗口改成另一组特征，复用现有事实，给新配方建立新身份；旧实验继续绑定原配方。

如果新增字段，先看 Raw 是否已经收到它。已有字段可以补合同/映射后重建所选域；缺少的响应再补对应接口和窗口。当前 rebuild 替换整个所选域，必须给它完整 Raw 集合，不能只给一天数据就期待自动补好整年。

若发现单位、映射、Reader 或复权错误，应留下错误范围、旧版本与替代版本，把真正用到这些输入的实验列为待复核，再重做受影响部分。保存旧结果是为了比较和追踪，不会让已知错误继续成为有效结论。

同样，后续供应商修订形成新观察和新 Snapshot；已保存的旧查询仍指向原版本。完整操作见[数据变化与恢复](../docs/data-change-and-recovery.md)，Raw 选择、半写恢复与 builder 绑定放在附录。


## 10. 选用 Qlib 时复用同一份导出

如果研究代码需要 Qlib，将已固定的 Snapshot 与查询显式导出，再由 Research 的 QlibView 读取。下面读取股票和 ETF 的已有导出。

导出时已经固定 cutoff。只改模型或用已有列换特征配方，可继续用同一目录；改数据版本、字段或 cutoff，建立新导出。原始财报仍通过事件查询处理。


In [35]:
sys.path.insert(0, str(WORKSPACE / 'axiom-research' / 'src'))
from axiom_research.qlib_adapter import QlibView
from axiom_data import verify_qlib_export
import numpy as np
qlib_report = json.loads((REPO/'delivery/qlib_actual_20261003.acceptance.json').read_text())
stock_qlib = QlibView(WORKSPACE/'data/qlib_views/stock_daily_202606_202608_v1').activate()
qlib_frame = stock_qlib.read(fields=('open','close','volume_shares','factor'),
                            start='2026-08-24', end='2026-08-31')
display(qlib_frame.head(8))
display(pd.DataFrame(stock_qlib.reference['fields']).T[['domain','field','unit','source_dtype']])
print('固定来源 Snapshot:', stock_qlib.reference['snapshot_id'])
print('QlibView:', stock_qlib.reference['view_id'])
print('导出时固定的 PIT:', stock_qlib.reference['queries'][0]['pit_policy'])
print('实际 Qlib 验收:', qlib_report['qlib_version'], qlib_report['views']['stock']['compared_numeric_cells'], '单元')
assert stock_qlib.reference['snapshot_id'] == UNIFIED_SNAPSHOT
verify_qlib_export(stock_qlib.path)


$open  $close  $volume_shares  \
instrument                 datetime                                    
CNSTOCK.000001.SZ.19910403 2026-08-24  11.38   11.56     119902496.0   
                           2026-08-25  11.57   11.59      99488112.0   
                           2026-08-26  11.55   11.73     116728880.0   
                           2026-08-27  11.70   11.59      97570168.0   
                           2026-08-28  11.54   11.65      83812624.0   
                           2026-08-31  11.64   11.72      90885656.0   
CNSTOCK.600000.SH.19991110 2026-08-24   9.01    9.22      79110608.0   
                           2026-08-25   9.24    9.08      88175624.0   

                                          $factor  
instrument                 datetime                
CNSTOCK.000001.SZ.19910403 2026-08-24  139.007996  
                           2026-08-25  139.007996  
                           2026-08-26  139.007996  
                           2026-08-27  139.007996  
                           2026-08-28  139.007996  
                           2026-08-31  139.007996  
CNSTOCK.600000.SH.19991110 2026-08-24   17.377399  
                           2026-08-25   17.377399

,domain,field,unit,source_dtype
amount_cny,market_daily,amount_cny,CNY,float64
close,market_daily,close,CNY/share,float64
down_limit,price_limits,down_limit,CNY/share,float64
factor,adjustment_factors,factor,dimensionless,float64
high,market_daily,high,CNY/share,float64
low,market_daily,low,CNY/share,float64
open,market_daily,open,CNY/share,float64
up_limit,price_limits,up_limit,CNY/share,float64
volume_shares,market_daily,volume_shares,shares,int64


固定来源 Snapshot: s_a0db6028118997d93b9f70dff852b701b030b152a30b2e8d325c08ad64931049
QlibView: f4ebb4297ab7fd892c72ac767207365b9894843fcdb262f7e21339a820790f52
导出时固定的 PIT: best_effort_vendor_v1
实际 Qlib 验收: 0.9.7 1170 单元


{'calendar': ['2026-06-01',
  '2026-06-02',
  '2026-06-03',
  '2026-06-04',
  '2026-06-05',
  '2026-06-08',
  '2026-06-09',
  '2026-06-10',
  '2026-06-11',
  '2026-06-12',
  '2026-06-15',
  '2026-06-16',
  '2026-06-17',
  '2026-06-18',
  '2026-06-22',
  '2026-06-23',
  '2026-06-24',
  '2026-06-25',
  '2026-06-26',
  '2026-06-29',
  '2026-06-30',
  '2026-07-01',
  '2026-07-02',
  '2026-07-03',
  '2026-07-06',
  '2026-07-07',
  '2026-07-08',
  '2026-07-09',
  '2026-07-10',
  '2026-07-13',
  '2026-07-14',
  '2026-07-15',
  '2026-07-16',
  '2026-07-17',
  '2026-07-20',
  '2026-07-21',
  '2026-07-22',
  '2026-07-23',
  '2026-07-24',
  '2026-07-27',
  '2026-07-28',
  '2026-07-29',
  '2026-07-30',
  '2026-07-31',
  '2026-08-03',
  '2026-08-04',
  '2026-08-05',
  '2026-08-06',
  '2026-08-07',
  '2026-08-10',
  '2026-08-11',
  '2026-08-12',
  '2026-08-13',
  '2026-08-14',
  '2026-08-17',
  '2026-08-18',
  '2026-08-19',
  '2026-08-20',
  '2026-08-21',
  '2026-08-24',
  '2026-08-25',
  '2026-08-2

ETF 读取保持基金份额单位和完整预热日历。命名成员文件表达当时可见的成员区间，`all.txt` 则是候选范围。下面的比较核对 Reader 与 Qlib 的键、NaN、单位和数值容差。


In [36]:
etf_qlib = QlibView(WORKSPACE/'data/qlib_views/etf_daily_202606_202608_v1').activate()
etf_native = etf_qlib.read(fields=('open','close','volume_units','factor'),
                          start='2026-08-24',end='2026-08-31')
display(etf_native.head(8))
print('基金成交量单位:', etf_qlib.reference['fields']['volume_units']['unit'])
print('含预热的完整日历:', len(etf_qlib.reference['calendar']), '交易日')
print('ETF 数值等价核对:', qlib_report['views']['etf']['compared_numeric_cells'], '单元')
print('训练、动量和策略 Feature 由 Research 显式定义；这里仅读取原始事实。')
verify_qlib_export(etf_qlib.path)


$open  $close  $volume_units  $factor
instrument                 datetime                                         
CN.ETF.SSE.510300.20120528 2026-08-24  4.678   4.627    922973824.0   1.2671
                           2026-08-25  4.601   4.616    745256832.0   1.2671
                           2026-08-26  4.615   4.652    846320832.0   1.2671
                           2026-08-27  4.659   4.691    641929728.0   1.2671
                           2026-08-28  4.684   4.679    604693312.0   1.2671
                           2026-08-31  4.640   4.685    673126464.0   1.2671
CN.ETF.SSE.510500.20130315 2026-08-24  7.859   7.740    352454464.0   0.3401
                           2026-08-25  7.663   7.726    378296064.0   0.3401

基金成交量单位: fund units
含预热的完整日历: 114 交易日
ETF 数值等价核对: 7182 单元
训练、动量和策略 Feature 由 Research 显式定义；这里仅读取原始事实。


{'calendar': ['2026-03-18',
  '2026-03-19',
  '2026-03-20',
  '2026-03-23',
  '2026-03-24',
  '2026-03-25',
  '2026-03-26',
  '2026-03-27',
  '2026-03-30',
  '2026-03-31',
  '2026-04-01',
  '2026-04-02',
  '2026-04-03',
  '2026-04-07',
  '2026-04-08',
  '2026-04-09',
  '2026-04-10',
  '2026-04-13',
  '2026-04-14',
  '2026-04-15',
  '2026-04-16',
  '2026-04-17',
  '2026-04-20',
  '2026-04-21',
  '2026-04-22',
  '2026-04-23',
  '2026-04-24',
  '2026-04-27',
  '2026-04-28',
  '2026-04-29',
  '2026-04-30',
  '2026-05-06',
  '2026-05-07',
  '2026-05-08',
  '2026-05-11',
  '2026-05-12',
  '2026-05-13',
  '2026-05-14',
  '2026-05-15',
  '2026-05-18',
  '2026-05-19',
  '2026-05-20',
  '2026-05-21',
  '2026-05-22',
  '2026-05-25',
  '2026-05-26',
  '2026-05-27',
  '2026-05-28',
  '2026-05-29',
  '2026-06-01',
  '2026-06-02',
  '2026-06-03',
  '2026-06-04',
  '2026-06-05',
  '2026-06-08',
  '2026-06-09',
  '2026-06-10',
  '2026-06-11',
  '2026-06-12',
  '2026-06-15',
  '2026-06-16',
  '2026-06-1

## 11. 附录：接口、版本与工程核对

前面的联合输入保存为 FeatureBuild；ETF 动量最终信号可直接交给 Engine 的中立 SignalFrame。Engine 保存意图、模拟成交和账户结果，UI 加载这些结果展示。

本轮七 ETF 的完整闭环见[当前交付](../docs/current-delivery.md)：默认严格状态政策保留零成交对照；用户批准的显式日线近似保留 UNKNOWN 与准入标记。教程下面的旧薄接口输出使用两个观察月末，Core return 是相邻观察间的变化；它们是接口示例，新的账户结果按 owner 报告读取。

这里集中保留来源、长标识、正式根不变检查和运行环境。需要复跑时再展开。


In [37]:
sys.path.insert(0, str(WORKSPACE / 'axiom-research' / 'src'))
sys.path.insert(0, str(WORKSPACE / 'axiom-engine' / 'src'))
sys.path.insert(0, str(WORKSPACE / 'axiom-ui' / 'src'))
from axiom_engine.runtime import DecisionBatchGate
from axiom_research.data_adapter import adapt_decision_batch
from axiom_ui import chart_context, project_data_layer, unavailable_layer

# Runtime pins one concrete Snapshot and records both domain queries.
# The unified Snapshot includes two-security market facts and Tushare membership snapshots.
# This is a best-effort vendor-snapshot protocol demonstration, not a strict
# historical 2026-08 decision: the membership dependencies arrived later.
gate = DecisionBatchGate(UNIFIED)
gate.pin_session(MONTH_ENDS[-1], snapshot=MEMBER_SNAPSHOT)
clock = MONTH_ENDS[-1] + 'T20:30:00+08:00'
fact_query = query(sessions=MONTH_ENDS)
facts = gate.read_decision(decision_session=MONTH_ENDS[-1], query=fact_query, clock=clock)
vendor_member_query = replace(query(('is_member',), sessions=MONTH_ENDS,
    domain='universe_membership'), universe_id='csi1800')

In [38]:
refs = gate.read_decision(decision_session=MONTH_ENDS[-1], query=vendor_member_query, clock=clock)
assert refs.frame.is_member.tolist() == [True,True]
print('成员查询 policy:', refs.context['query']['pit_policy'],
      '；历史成员限制:', refs.context['limitations'])
recipe = 'sha256:' + hashlib.sha256(b'real month-end identity-lag-return tutorial v1').hexdigest()
adapted = adapt_decision_batch(facts, reference=refs, recipe_ref=recipe)
core_output = adapted.execute().to_dict()
display(pd.DataFrame(core_output['rows']))
print('Core输入单位:', adapted.plan.to_dict()['input_schema'][0]['unit'])
assert adapted.plan.to_dict()['input_schema'][0]['unit'] == 'CNY/share'
assert len(gate.references) == 2
# This return spans TWO OBSERVED MONTH-ENDS, not two consecutive trading days.
raw_close = facts.frame.close.tolist()
assert abs(core_output['rows'][1]['values'][2] - (raw_close[1]/raw_close[0]-1)) < 1e-12
view_for_ui = adapted.view_ref.to_dict()

成员查询 policy: best_effort_vendor_v1 ；历史成员限制: ['vendor release time is an assumption, not historical vintage evidence', 'Supplier index_weight was observed on dated snapshots only; no intramonth membership change is proven.', 'Best-effort uses an assumed 18:00 Asia/Shanghai dated release; strict policies use actual Raw receipt.']


,availability,cutoff,reasons,security_id,session,sources,valid,values
0,"[2026-07-31T12:00:00Z, None, 2026-07-31T12:00:...",2026-07-31T12:30:00Z,"[[], [INSUFFICIENT_HISTORY], [MISSING]]",cnstock.000001.SZ.19910403,2026-07-31,[[sha256:d730a0d0410cb729b0ad9ca3ec5ff2e38e66e...,"[True, False, False]","[11.63, None, None]"
1,"[2026-08-31T12:00:00Z, 2026-07-31T12:00:00Z, 2...",2026-08-31T12:30:00Z,"[[], [], []]",cnstock.000001.SZ.19910403,2026-08-31,[[sha256:bcf4678dadf562e45127ded94742996bdb540...,"[True, True, True]","[11.72, 11.63, 0.007738607050730817]"


Core输入单位: CNY/share


In [39]:
chart = chart_context(mode='run_replay', security_id=SYMBOL,
    session_refs={d: view_for_ui for d in MONTH_ENDS}, price_basis='unadjusted',
    adjustment_anchor=None, time_axis='decision_available')
layer = project_data_layer(facts, field='close', security_id=SYMBOL, role='data_fact', chart=chart)
display(pd.DataFrame(layer['points']))
assert [p['value'] for p in layer['points']] == raw_close
print('Data wire:', list(facts.to_response()))
print('UI协议:', layer['contract_version'], '固定版本:', layer['source_ref']['snapshot_id'])
print('无成交产物时 UI 明确 unavailable:', unavailable_layer(role='fill', requested_ref=None))
# Replay-side access is explicitly separate from decision-side facts.
market_query = replace(fact_query, purpose='market_replay')
market = gate.read_market(decision_session=MONTH_ENDS[-1], query=market_query, clock=clock)
assert market.context['query']['purpose'] == 'market_replay'

,security_id,session,value,provenance,missing_reason,source_available_time,observed_time
0,cnstock.000001.SZ.19910403,2026-07-31,11.63,"{'security_id': 'cnstock.000001.SZ.19910403', ...",None,2026-07-31T20:00:00+08:00,2026-09-28T15:06:01.854164+00:00
1,cnstock.000001.SZ.19910403,2026-08-31,11.72,"{'security_id': 'cnstock.000001.SZ.19910403', ...",None,2026-08-31T20:00:00+08:00,2026-09-28T15:06:14.947033+00:00


Data wire: ['records', 'field_meta', 'context']
UI协议: p12_chart_projection_v1 固定版本: s_a0db6028118997d93b9f70dff852b701b030b152a30b2e8d325c08ad64931049
无成交产物时 UI 明确 unavailable: {'contract_version': 'p12_chart_projection_v1', 'role': 'fill', 'requested_ref': None, 'requested_stage': None, 'unit': None, 'status': 'LAYER_UNAVAILABLE', 'points': None, 'is_reconstructed': False, 'generated_at': '2026-10-03T15:04:40.968972+00:00'}


### 运行环境与数据位置

两份 ipynb 是教学源，HTML 从保存输出生成。离线初始化使用 [tutorial_support.py](../examples/tutorial_support.py)；Data 根与留存报告在本地，路径可通过 `AXIOM_WORKSPACE`、`AXIOM_TUTORIAL_DATA_ROOT`、`AXIOM_UNIFIED_DATA_ROOT` 调整。

复跑入口及依赖见[教程说明](README.md)。涉及 Data 构建时，用 `AXIOM_TUTORIAL_DATA_PACKAGE` 绑定实际留存安装包；代码版本、wheel 和此前执行范围见[versions.json](../versions.json)。下面保留目录大小及核对结果，耗时仅属于对应样本。


In [40]:
sizes = pd.DataFrame([{'component': name, **measure_tree(ROOT/name)}
    for name in ('raw', 'canonical', 'snapshots', 'operations')])
display(sizes)
print('工作数据根:', measure_tree(ROOT))
print('当前搬移包:', measure_tree(Path(str(ROOT)+'-bundle')))
print('此前真实运行计时（非本次重新联网）:', RUN['timing'])
print('12×一年 active runtime:', round(RUN['timing']['bulk_active_process_seconds']*12/60, 1), '分钟')
print('12×一年目录 logical bytes:', round(measure_tree(ROOT)['bytes']*12/2**30, 2), 'GiB')


,component,files,bytes,MiB
0,raw,1791,252739100,241.03
1,canonical,70,44813913,42.74
2,snapshots,20,36544652,34.85
3,operations,33,1059911,1.01


工作数据根: {'files': 1916, 'bytes': 335157660, 'MiB': 319.63}
当前搬移包: {'files': 835, 'bytes': 252134085, 'MiB': 240.45}
此前真实运行计时（非本次重新联网）: {'bulk_active_process_seconds': 193.56, 'bulk_wall_including_fix_pause_seconds': 282.616, 'bulk_peak_rss_bytes': 327614464, 'offline_full_rebuild_seconds': 38.59, 'audit_seconds': 10.864, 'note': 'Two resumed real processes; no clean uninterrupted full-year or 12-year timing claim.'}
12×一年 active runtime: 38.7 分钟
12×一年目录 logical bytes: 3.75 GiB


In [41]:
full_sizes = pd.DataFrame([{'component':name, **measure_tree(FULL_ROOT/name)}
    for name in ('raw','canonical','snapshots','operations')])
display(full_sizes)
print('完整源小样本根合计:', measure_tree(FULL_ROOT))
print('这是两证券三个月及财务 warmup 的现场字节，不外推为十二年全市场实测。')


,component,files,bytes,MiB
0,raw,794,126359144,120.51
1,canonical,58,834293,0.80
2,snapshots,28,59176344,56.43
3,operations,81,887956,0.85


完整源小样本根合计: {'files': 963, 'bytes': 187257821, 'MiB': 178.58}
这是两证券三个月及财务 warmup 的现场字节，不外推为十二年全市场实测。


### 时间口径与来源范围

股票行情、两证券完整来源、七 ETF 是三份固定数据。两证券教学样本回放留存 Tushare Raw，receipt 是回放时的实收时间；另外的在线采集与搬移证据见[preflight](../docs/bulk-preflight.md)。

历史终态与供应商日期假设、源字段/单位核对、模型/OOS、执行能力各自回答不同问题。十二年 bulk、跨 OS、多年性能与完整 Runtime/UI 的现状由[当前交付](../docs/current-delivery.md)统一维护。


In [42]:
assert DATA.resolve('current') == CURRENT_BEFORE
assert ((UNIFIED_ROOT/'current.json').read_bytes()
    if (UNIFIED_ROOT/'current.json').exists() else None) == UNIFIED_CURRENT_BEFORE
assert (ETF_ROOT/'current.json').read_bytes() == ETF_CURRENT_BEFORE
print('正式数据根的 current 未变化；教程无供应商调用。临时文件即将清理。')
TEMPDIR.cleanup()


正式数据根的 current 未变化；教程无供应商调用。临时文件即将清理。
